In [ ]:

!pip -q install -U langchain langchain-community langchain-google-genai pymupdf neo4j pillow pytesseract opencv-python-headless
!sudo apt-get install -qq tesseract-ocr tesseract-ocr-vie


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests=

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 15.6 MB/s eta 0:00:00


In [ ]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.5 MB/s eta 0:00:00


In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 34.5 MB/s eta 0:00:00


In [ ]:
import os
import time
import json
import re
import uuid
import getpass
import hashlib
import base64
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime

import pandas as pd
import numpy as np
from IPython.display import Markdown, display

import fitz  # PyMuPDF
from PIL import Image
import cv2
import pytesseract

from google.colab import files, userdata
from langchain_community.document_loaders import PyPDFDirectoryLoader
from neo4j import GraphDatabase
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
import google.generativeai as genai

print("Đã import tất cả thư viện")

/tmp/ipykernel_1991/1762353053.py:23: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Đã import tất cả thư viện


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================
# CELL 3: LẤY SECRETS VÀ CẤU HÌNH
# ============================================
def get_secret(secret_name: str):
    """Lấy secret từ Google Colab nếu có; nếu không, yêu cầu nhập thủ công."""
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(secret_name)
    except Exception:
        value = os.environ.get(secret_name)
    if not value:
        value = getpass.getpass(f"Nhập {secret_name}: ")
    return value

# Lấy API keys
GOOGLE_API_KEY = get_secret("APIDengue")
GROQ_API_KEY = get_secret("GROQ_API_KEY")
NEO4J_URI = get_secret("NEO4J_URI")
NEO4J_USERNAME = get_secret("NEO4J_USERNAME")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE")

print(" Đã nhận tất cả API keys. Không in key ra màn hình.")

# Cấu hình
EMBEDDING_MODEL = "models/gemini-embedding-001"
EMBEDDING_DIMENSION = 768
CHUNK_SIZE = 800
CHUNK_OVERLAP = 400
LLM_MODEL = "gemini-1.5-pro"

# Đường dẫn
DOCUMENT_DIR = Path("/content/drive/MyDrive/Document")
IMAGES_DIR = Path("/content/drive/MyDrive/Images")
CHUNKS_DIR = Path("/content/drive/MyDrive/Chunks")
DOCUMENT_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)
CHUNKS_DIR.mkdir(exist_ok=True)

 Đã nhận tất cả API keys. Không in key ra màn hình.


In [ ]:

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=GOOGLE_API_KEY,
    output_dimensionality=EMBEDDING_DIMENSION
)

# Kiểm tra chiều vector embedding
test_vector = embeddings.embed_query("Kiểm tra chiều vector embedding cho GraphRAG.")
print("Embedding model:", EMBEDDING_MODEL)
print("Embedding dimension:", len(test_vector))
assert len(test_vector) == EMBEDDING_DIMENSION, (
    f"Embedding dimension không khớp: {len(test_vector)} != {EMBEDDING_DIMENSION}"
)
print("Embedding dimension OK")

Embedding model: models/gemini-embedding-001
Embedding dimension: 768
Embedding dimension OK


In [ ]:
def connect_neo4j():
    """Kết nối đến Neo4j AuraDB"""
    try:
        driver = GraphDatabase.driver(
            NEO4J_URI,
            auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
        )
        driver.verify_connectivity()
        print("Đã kết nối Neo4j!")
        return driver
    except Exception as e:
        print(f"Lỗi kết nối Neo4j: {e}")
        return None

driver = connect_neo4j()

Đã kết nối Neo4j!


In [ ]:
def create_graph_schema(driver):
    """Tạo schema cho Knowledge Graph trong Neo4j"""
    with driver.session() as session:
        # Constraints cho MedicalChunk
        session.run("""
            CREATE CONSTRAINT medical_chunk_id_unique
            IF NOT EXISTS
            FOR (c:MedicalChunk) REQUIRE c.chunk_id IS UNIQUE
        """)

        # Constraints cho Entity
        session.run("""
            CREATE CONSTRAINT entity_id_unique
            IF NOT EXISTS
            FOR (e:Entity) REQUIRE e.id IS UNIQUE
        """)

        # Constraints cho Image
        session.run("""
            CREATE CONSTRAINT image_path_unique
            IF NOT EXISTS
            FOR (i:Image) REQUIRE i.path IS UNIQUE
        """)

        # Indexes
        session.run("""
            CREATE INDEX medical_chunk_text_idx
            IF NOT EXISTS
            FOR (c:MedicalChunk) ON (c.text)
        """)
        session.run("""
            CREATE INDEX medical_chunk_page_idx
            IF NOT EXISTS
            FOR (c:MedicalChunk) ON (c.page_num)
        """)
        session.run("""
            CREATE FULLTEXT INDEX chunk_text_index
            IF NOT EXISTS
            FOR (c:MedicalChunk)
            ON EACH [c.text]
        """)

        # Vector index cho entity embeddings
        session.run("""
            CREATE VECTOR INDEX entity_embeddings IF NOT EXISTS
            FOR (e:Entity) ON (e.embedding)
            OPTIONS {indexConfig: {
                `vector.dimensions`: 768,
                `vector.similarity_function`: 'cosine'
            }}
        """)
        print(" Đã tạo Graph schema")
create_graph_schema(driver)

In [ ]:
# ============================================
# CELL 9: ĐỌC PDF VÀ PHÂN BIỆT TEXT - BẢNG (CẢI TIẾN)
# ============================================
import pdfplumber
import re
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def extract_text_and_tables_from_pdf(pdf_path: str):
    """
    Đọc PDF, phân biệt text và bảng bằng pdfplumber
    Trả về: Danh sách các chunk (text và table)
    """
    chunks = []

    print(f" Đang đọc PDF: {pdf_path}")

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        print(f" Tổng số trang: {total_pages}")

        for page_num, page in enumerate(pdf.pages, start=1):

            # 1. TRÍCH XUẤT BẢNG
            tables = page.extract_tables()

            if tables:
                for table_idx, table in enumerate(tables):
                    if table and len(table) > 1:  # Kiểm tra bảng có dữ liệu
                        # Chuyển bảng thành markdown hoặc text có cấu trúc
                        table_text = convert_table_to_markdown(table)

                        chunks.append({
                            "page": page_num,
                            "type": "table",
                            "table_idx": table_idx,
                            "content": table_text,
                            "raw_table": table  # Lưu raw để xử lý sau nếu cần
                        })

            # 2. TRÍCH XUẤT VĂN BẢN (loại bỏ phần đã trùng với bảng)
            # Lấy text toàn trang, sau đó loại bỏ các vùng bảng
            full_text = page.extract_text()

            if full_text:
                # Loại bỏ các dòng trùng với bảng đã trích xuất
                # (Cách đơn giản: lấy text thô)
                text_content = clean_text(full_text)

                chunks.append({
                    "page": page_num,
                    "type": "text",
                    "content": text_content
                })

            # Hiển thị tiến độ
            if page_num % 10 == 0 or page_num == total_pages:
                print(f"   Đã xử lý trang {page_num}/{total_pages}")

    print(f"\n Đã phân tích PDF:")
    print(f"   Tổng số chunk: {len(chunks)}")

    # Thống kê
    text_count = sum(1 for c in chunks if c['type'] == 'text')
    table_count = sum(1 for c in chunks if c['type'] == 'table')
    print(f"   Text chunks: {text_count}")
    print(f"   Table chunks: {table_count}")

    # Hiển thị ví dụ
    if chunks:
        print("\n Ví dụ text:")
        for chunk in chunks:
            if chunk['type'] == 'text':
                content_preview = chunk['content'][:200] + "..." if len(chunk['content']) > 200 else chunk['content']
                print(f"   Trang {chunk['page']}: {content_preview}")
                break

        print("\n Ví dụ bảng:")
        for chunk in chunks:
            if chunk['type'] == 'table':
                content_preview = chunk['content'][:200] + "..." if len(chunk['content']) > 200 else chunk['content']
                print(f"   Trang {chunk['page']}: {content_preview}")
                break

    return chunks

def convert_table_to_markdown(table):
    """Chuyển đổi bảng sang định dạng Markdown để dễ đọc và xử lý"""
    if not table or len(table) == 0:
        return ""

    # Xác định headers (dòng đầu tiên)
    headers = table[0] if table else []
    headers = [str(h).strip() if h else "" for h in headers]

    # Xác định dữ liệu (các dòng còn lại)
    rows = table[1:] if len(table) > 1 else []

    # Tạo chuỗi markdown
    markdown_lines = []

    # Header
    if headers:
        markdown_lines.append("| " + " | ".join(headers) + " |")
        markdown_lines.append("|" + "|".join(["---" for _ in headers]) + "|")

    # Dữ liệu
    for row in rows:
        # Đảm bảo row có đủ số cột
        row_data = [str(cell).strip() if cell else "" for cell in row]
        while len(row_data) < len(headers):
            row_data.append("")
        markdown_lines.append("| " + " | ".join(row_data[:len(headers)]) + " |")

    return "\n".join(markdown_lines)



def clean_text(text):
    """Làm sạch văn bản"""
    if not text:
        return ""

    # Xóa khoảng trắng thừa
    text = re.sub(r'\n\s*\n', '\n\n',
                  text)
    text = re.sub(r' +', ' ', text)
    # Xóa các dòng trống đầu/cuối
    text = text.strip()

    return text



def chunk_documents(chunks, chunk_size=800, chunk_overlap=400):
    """
    Chia nhỏ các chunk text và table bằng RecursiveCharacterTextSplitter
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
        add_start_index=True
    )

    chunked_docs = []

    for chunk in chunks:
        if chunk['type'] == 'table':
            # Bảng không nên chia nhỏ quá nhiều
            # Tạo Document riêng cho mỗi bảng
            doc = Document(
                page_content=chunk['content'],
                metadata={
                    "page": chunk['page'],
                    "type": "table",
                    "table_idx": chunk.get('table_idx', 0)
                }
            )
            chunked_docs.append(doc)
        else:
            # Text: chia nhỏ bằng RecursiveCharacterTextSplitter
            split_docs = text_splitter.split_text(chunk['content'])

            for i, text_chunk in enumerate(split_docs):
                doc = Document(
                    page_content=text_chunk,
                    metadata={
                        "page": chunk['page'],
                        "type": "text",
                        "chunk_index": i,
                        "total_chunks": len(split_docs)
                    }
                )
                chunked_docs.append(doc)

    print(f"\n Đã chia thành {len(chunked_docs)} chunks")
    return chunked_docs

# ============================================
# SỬ DỤNG
# ============================================

# Đọc PDF và trích xuất
if pdf_files:
    pdf_path = str(pdf_files[0])

    # Bước 1: Trích xuất text và bảng
    raw_chunks = extract_text_and_tables_from_pdf(pdf_path)

    # Bước 2: Chia nhỏ bằng RecursiveCharacterTextSplitter
    chunk_size = 1000  # Có thể điều chỉnh
    chunk_overlap = 200
    final_chunks = chunk_documents(raw_chunks, chunk_size, chunk_overlap)

    # Bước 3: Lưu kết quả để sử dụng
    print(f"\n✅ Hoàn tất! Có {len(final_chunks)} chunks sẵn sàng để embedding.")

    # Hiển thị ví dụ
    if final_chunks:
        print("\n Ví dụ chunk đầu tiên:")
        first_chunk = final_chunks[0]
        print(f"   Loại: {first_chunk.metadata.get('type', 'unknown')}")
        print(f"   Trang: {first_chunk.metadata.get('page', 'unknown')}")
        print(f"   Nội dung: {first_chunk.page_content[:200]}...")

else:
    print(" Không có file PDF")
    raw_chunks = []
    final_chunks = []

In [ ]:
# ============================================
# CELL 11: TRÍCH XUẤT ENTITY CHO SỐT XUẤT HUYẾT DENGUE
# ============================================
import json
import re
import time
from typing import Dict, List
from collections import Counter
from langchain_core.documents import Document
from langchain_groq import ChatGroq

# ============================================
# 1. ĐỊNH NGHĨA SCHEMA CHO SỐT XUẤT HUYẾT DENGUE
# ============================================

ENTITY_TYPES = [
    "benh",             # Bệnh: Sốt xuất huyết Dengue
    "thuoc",            # Thuốc: Paracetamol, Ringerlactate, NaCl, CPT
    "trieu_chung",      # Triệu chứng: sốt, đau đầu, xuất huyết
    "lieu_dung",
    "muc_do",           # Mức độ: nhẹ, cảnh báo, nặng, sốc
    "nhom_benh_nhan",   # Nhóm bệnh nhân: trẻ em, người lớn, phụ nữ có thai
    "dieu_tri",         # Phương pháp điều trị: nhập viện, thở oxy, truyền dịch
    "chi_dinh",
    "chong_chi_dinh",
    "xet_nghiem"
]

RELATIONSHIP_TYPES = [
    "dieu_tri",
    "co_trieu_chung",
    "co_lieu_dung",
    "chong_chi_dinh",
    "chi_dinh",
    "co_nguy_co",
    "chan_doan",
    "phong_ngua"
]

# ============================================
# 2. TỪ ĐIỂN CHUYÊN NGÀNH SỐT XUẤT HUYẾT DENGUE
# ============================================

# Tên bệnh
BENH_NAMES = [
    "sốt xuất huyết dengue",
    "sốt xuất huyết",
    "sxhd",
    "dengue"
]

# Thuốc và dịch truyền
THUOC_NAMES = [
    "paracetamol",
    "ringer lactate",
    "ringer acetate",
    "nacl 0,9%",
    "natri clorua",
    "dextran",
    "hes",
    "albumin",
    "dopamin",
    "dobutamin",
    "noradrenalin",
    "adrenalin",
    "furosemide",
    "vitamin k",
    "omeprazole",
    "cao phân tử",
    "dịch truyền"
]

# Triệu chứng
TRIEU_CHUNG_NAMES = [
    "sốt", "sốt cao", "sốt đột ngột", "sốt liên tục",
    "đau đầu", "nhức đầu",
    "đau cơ", "đau khớp",
    "buồn nôn", "nôn", "nôn ói", "nôn nhiều",
    "phát ban",
    "xuất huyết da", "xuất huyết niêm mạc",
    "chảy máu chân răng", "chảy máu mũi", "chảy máu cam",
    "vật vã", "lừ đừ", "li bì", "bứt rứt",
    "đau bụng", "đau bụng nhiều",
    "gan to", "tiểu ít",
    "mạch nhanh", "mạch chậm",
    "huyết áp tụt", "huyết áp kẹt",
    "sốc", "khó thở", "suy hô hấp",
    "tràn dịch màng phổi", "tràn dịch màng bụng",
    "vàng da", "rối loạn tri giác", "hôn mê", "co giật"
]

# Nhóm bệnh nhân
NHOM_BENH_NHAN = [
    "trẻ nhũ nhi",
    "trẻ em",
    "trẻ vị thành niên",
    "người lớn",
    "người cao tuổi",
    "phụ nữ có thai",
    "trẻ béo phì",
    "bệnh nhân thalassemia"
]

# Mức độ bệnh
MUC_DO_BENH = [
    "không có dấu hiệu cảnh báo",
    "có dấu hiệu cảnh báo",
    "sốt xuất huyết nặng",
    "sốc",
    "sốc nặng"
]

# Phương pháp điều trị
DIEU_TRI_NAMES = [
    "điều trị ngoại trú",
    "nhập viện",
    "nhập khoa hồi sức",
    "nhập khoa cấp cứu",
    "hồi sức tích cực",
    "chống sốc",
    "truyền dịch",
    "thở oxy",
    "thở máy",
    "lọc máu",
    "truyền máu"
]

# Xét nghiệm
XET_NGHIEM_NAMES = [
    "hct",
    "hematocrit",
    "tiểu cầu",
    "ast",
    "alt",
    "ns1",
    "igm",
    "igg",
    "pcr"
]

# ============================================
# 3. HÀM TRÍCH XUẤT ENTITY
# ============================================

def extract_entities_from_chunk(text: str, chunk_id: str, chunk_type: str, chunk_page: int) -> Dict:
    """Trích xuất entity và relationship từ chunk bằng Groq"""

    # Kiểm tra nếu chunk chứa thông tin về Sốt xuất huyết
    text_lower = text.lower()
    is_dengue = any(benh in text_lower for benh in BENH_NAMES)

    if not is_dengue:
        return {"entities": [], "relationships": [], "chunk_id": chunk_id}

    prompt = f"""
Bạn là chuyên gia y tế về bệnh Sốt xuất huyết Dengue. Phân tích đoạn văn sau và trích xuất thông tin:

**ĐOẠN VĂN:**
{text}

**CHỈ TRÍCH XUẤT THÔNG TIN VỀ SỐT XUẤT HUYẾT DENGUE**

**PHÂN LOẠI ENTITY:**
- benh: bệnh (Sốt xuất huyết Dengue)
- thuoc: thuốc, dịch truyền (Paracetamol, Ringerlactate, NaCl, CPT)
- trieu_chung: triệu chứng (sốt, đau đầu, xuất huyết)
- lieu_dung: liều dùng (10ml/kg, 500mg)
- muc_do: mức độ (nhẹ, cảnh báo, nặng, sốc)
- nhom_benh_nhan: nhóm bệnh nhân (trẻ em, người lớn, phụ nữ có thai)
- dieu_tri: phương pháp điều trị (nhập viện, thở oxy, truyền dịch)
- chi_dinh: chỉ định
- chong_chi_dinh: chống chỉ định
- xet_nghiem: xét nghiệm (Hct, tiểu cầu, AST/ALT)

**PHÂN LOẠI RELATIONSHIP:**
- dieu_tri: điều trị
- co_trieu_chung: có triệu chứng
- co_lieu_dung: có liều dùng
- chong_chi_dinh: chống chỉ định
- chi_dinh: chỉ định
- co_nguy_co: có yếu tố nguy cơ

**YÊU CẦU:**
- Chỉ trích xuất thông tin về Sốt xuất huyết Dengue
- Giữ nguyên tiếng Việt
- Nếu không có, trả về mảng rỗng

**TRẢ VỀ JSON DUY NHẤT:**
{{
    "entities": [
        {{"name": "Sốt xuất huyết Dengue", "type": "benh"}},
        {{"name": "Paracetamol", "type": "thuoc"}},
        {{"name": "Sốt cao", "type": "trieu_chung"}}
    ],
    "relationships": [
        {{"source": "Sốt xuất huyết Dengue", "target": "Sốt cao", "type": "co_trieu_chung"}},
        {{"source": "Paracetamol", "target": "Sốt xuất huyết Dengue", "type": "dieu_tri"}}
    ]
}}
"""

    try:
        response = llm.invoke(prompt)
        content = response.content.strip()

        json_match = re.search(r'\{.*\}', content, re.DOTALL)
        if json_match:
            json_str = json_match.group()
            json_str = re.sub(r',\s*}', '}', json_str)
            json_str = re.sub(r',\s*]', ']', json_str)
            result = json.loads(json_str)

            if "entities" not in result:
                result["entities"] = []
            if "relationships" not in result:
                result["relationships"] = []

            result["chunk_id"] = chunk_id
            result["type"] = chunk_type
            result["page"] = chunk_page

            return result
        else:
            return {"entities": [], "relationships": [], "chunk_id": chunk_id}

    except Exception as e:
        print(f"   Loi chunk {chunk_id}: {e}")
        return {"entities": [], "relationships": [], "chunk_id": chunk_id}

# ============================================
# 4. HÀM XỬ LÝ ALL CHUNKS
# ============================================

def process_all_chunks(docs: List[Document], verbose: bool = True) -> Dict:
    """Xử lý tất cả chunks để trích xuất entities và relationships"""

    total_chunks = len(docs)
    print(f"Bat dau xu ly {total_chunks} chunks (chi lay thong tin Sot xuat huyet Dengue)")
    print("=" * 60)

    all_entities = []
    all_relationships = []
    chunk_stats = {
        "total": total_chunks,
        "processed": 0,
        "has_entities": 0,
        "has_relationships": 0,
        "skip_non_dengue": 0
    }

    entity_type_counter = Counter()
    relationship_type_counter = Counter()

    for i, doc in enumerate(docs):
        chunk_id = doc.metadata.get("chunk_id", f"chunk_{i}")
        chunk_type = doc.metadata.get("type", "text")
        chunk_page = doc.metadata.get("page", 0)

        print(f"\n[Chunk {i+1}/{total_chunks}] ID: {chunk_id} | Trang: {chunk_page} | Loai: {chunk_type}")

        content_preview = doc.page_content[:100].replace('\n', ' ') + "..."
        print(f"   Noi dung: {content_preview}")

        # Kiểm tra nếu chunk chứa thông tin về Sốt xuất huyết
        text_lower = doc.page_content.lower()
        is_dengue = any(benh in text_lower for benh in BENH_NAMES)

        if not is_dengue:
            chunk_stats["skip_non_dengue"] += 1
            print(f"   Bo qua: khong phai thong tin ve Sot xuat huyet")
            continue

        print(f"   Dang trich xuat entity...")
        result = extract_entities_from_chunk(
            doc.page_content, chunk_id, chunk_type, chunk_page
        )

        chunk_stats["processed"] += 1

        entities = result.get("entities", [])
        if entities:
            chunk_stats["has_entities"] += 1
            print(f"   Tim thay {len(entities)} entities")
            for e in entities:
                e["source_chunk"] = chunk_id
                e["source_type"] = chunk_type
                e["page"] = chunk_page
                all_entities.append(e)
                entity_type_counter[e.get("type", "unknown")] += 1
        else:
            print(f"   Khong tim thay entity")

        relationships = result.get("relationships", [])
        if relationships:
            chunk_stats["has_relationships"] += 1
            print(f"   Tim thay {len(relationships)} relationships")
            for r in relationships:
                r["source_chunk"] = chunk_id
                r["source_type"] = chunk_type
                r["page"] = chunk_page
                all_relationships.append(r)
                relationship_type_counter[r.get("type", "unknown")] += 1
        else:
            print(f"   Khong tim thay relationship")

        progress_pct = (i + 1) / total_chunks * 100
        print(f"   Tien do: {i+1}/{total_chunks} ({progress_pct:.1f}%) | Entities: {len(all_entities)} | Relations: {len(all_relationships)}")

        time.sleep(0.1)

    # ============================================
    # THỐNG KÊ
    # ============================================
    print("\n" + "=" * 60)
    print("THONG KE TRICH XUAT ENTITY - SOT XUAT HUYET DENGUE")
    print("=" * 60)
    print(f"   Tong chunks: {chunk_stats['processed']}")
    print(f"   Bo qua (khong phai SXHD): {chunk_stats['skip_non_dengue']}")
    print(f"   Chunks co entities: {chunk_stats['has_entities']}")
    print(f"   Chunks co relationships: {chunk_stats['has_relationships']}")
    print(f"   Tong entities: {len(all_entities)}")
    print(f"   Tong relationships: {len(all_relationships)}")

    if entity_type_counter:
        print("\nPHAN BO ENTITY TYPES:")
        for entity_type, count in entity_type_counter.most_common():
            print(f"   {entity_type}: {count}")

    if all_entities:
        print("\nVI DU ENTITY (5 cai dau):")
        for entity in all_entities[:5]:
            print(f"   - {entity['name']} ({entity['type']}) - Trang {entity['page']}")

    if all_relationships:
        print("\nVI DU RELATIONSHIP (5 cai dau):")
        for rel in all_relationships[:5]:
            print(f"   - {rel['source']} --({rel['type']})--> {rel['target']}")

    return {
        "entities": all_entities,
        "relationships": all_relationships,
        "stats": {
            "chunk_stats": chunk_stats,
            "entity_types": dict(entity_type_counter),
            "relationship_types": dict(relationship_type_counter)
        }
    }


def deduplicate_entities(entities: List[Dict]) -> List[Dict]:
    """Loại bỏ entity trùng lặp"""
    seen = set()
    unique = []
    for e in entities:
        key = (e.get('name', '').lower().strip(), e.get('type', '').lower().strip())
        if key not in seen:
            seen.add(key)
            unique.append(e)
    print(f"Loai bo trung lap entities: {len(entities)} -> {len(unique)}")
    return unique

def deduplicate_relationships(relationships: List[Dict]) -> List[Dict]:
    """Loại bỏ relationship trùng lặp"""
    seen = set()
    unique = []
    for r in relationships:
        key = (
            r.get('source', '').lower().strip(),
            r.get('target', '').lower().strip(),
            r.get('type', '').lower().strip()
        )
        if key not in seen:
            seen.add(key)
            unique.append(r)
    print(f"Loai bo trung lap relationships: {len(relationships)} -> {len(unique)}")
    return unique



try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except:
    import getpass
    GROQ_API_KEY = getpass.getpass("Nhap GROQ_API_KEY: ")

llm = ChatGroq(
    model="llama-3.1-8b-instant",

    temperature=0.1,
    groq_api_key=GROQ_API_KEY
)

print("Da khoi tao Groq voi model: llama-3.1-8b-instant")

# Test thử
test_response = llm.invoke("Xin chao, hay gioi thieu ngan gon ve ban")
print(f"Test: {test_response.content[:100]}...\n")

# ============================================
# 7. CHẠY CHÍNH
# ============================================

if 'final_chunks' in globals() and final_chunks:
    print("BAT DAU TRICH XUAT ENTITY CHO SOT XUAT HUYET DENGUE")
    print("=" * 60)
    print(f"So chunks: {len(final_chunks)}")
    print("=" * 60)

    extracted_data = process_all_chunks(final_chunks, verbose=True)

    print("\nLOAI BO TRUNG LAP...")
    extracted_data["entities"] = deduplicate_entities(extracted_data["entities"])
    extracted_data["relationships"] = deduplicate_relationships(extracted_data["relationships"])

    print("\nHOAN TAT!")
    print(f"   Entities cuoi cung: {len(extracted_data['entities'])}")
    print(f"   Relationships cuoi cung: {len(extracted_data['relationships'])}")

    # Lưu kết quả
    with open('extracted_entities_sxhd.json', 'w', encoding='utf-8') as f:
        json.dump(extracted_data, f, ensure_ascii=False, indent=2)
    print("   Da luu vao extracted_entities_sxhd.json")

else:
    print("Khong co chunks de xu ly")
    print("   Kiem tra: final_chunks co ton tai va khong rong?")
    extracted_data = {"entities": [], "relationships": [], "stats": {}}

Da khoi tao Groq voi model: llama-3.1-8b-instant
Test: Xin chào! Tôi là một mô hình ngôn ngữ AI được thiết kế để hỗ trợ và cung cấp thông tin. Tôi có thể g...

BAT DAU TRICH XUAT ENTITY CHO SOT XUAT HUYET DENGUE
So chunks: 260
Bat dau xu ly 260 chunks (chi lay thong tin Sot xuat huyet Dengue)

[Chunk 1/260] ID: chunk_0 | Trang: 1 | Loai: table
   Noi dung: |  | 95/2022/NĐ-CP ngày 15 tháng 11 năm 2022 của | |---|---| | Chính phủ quy định chức năng, nhiệm v...
   Bo qua: khong phai thong tin ve Sot xuat huyet

[Chunk 2/260] ID: chunk_1 | Trang: 1 | Loai: text
   Noi dung: BỘ Y TẾ CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM 0 Độc lập - Tự do - Hạnh phúc 3 0: 3 Số: /QĐ-BY1T4: Hà Nộ...
   Dang trich xuat entity...
   Tim thay 3 entities
   Tim thay 2 relationships
   Tien do: 2/260 (0.8%) | Entities: 3 | Relations: 2

[Chunk 3/260] ID: chunk_2 | Trang: 1 | Loai: text
   Noi dung: Điều 2. Hướng dẫn chẩn đoán, điều trị Sốt xuất huyết Dengue được áp dụng tại các cơ sở khám bệnh, ch...
   Dang trich xua

In [ ]:
# ============================================
# CELL 16: HIEN THI ENTITIES VA RELATIONSHIPS TRONG NEO4J
# ============================================

import os
import getpass
from neo4j import GraphDatabase
from collections import Counter
import pandas as pd
from IPython.display import display, HTML

print("=" * 60)
print("HIEN THI ENTITIES VA RELATIONSHIPS TRONG NEO4J")
print("=" * 60)

# ============================================
# 1. LAY THONG TIN KET NOI
# ============================================

def get_secret(secret_name: str):
    """Lấy secret từ Google Colab nếu có; nếu không, yêu cầu nhập thủ công."""
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(secret_name)
    except Exception:
        value = os.environ.get(secret_name)
    if not value:
        value = getpass.getpass(f"Nhập {secret_name}: ")
    return value

NEO4J_URI = get_secret("NEO4J_URI")
NEO4J_USERNAME = get_secret("NEO4J_USERNAME")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE")

# ============================================
# 2. KET NOI NEO4J
# ============================================

print("\n[1] KET NOI NEO4J")
print("-" * 40)

try:
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
    )
    driver.verify_connectivity()
    print("Ket noi Neo4j thanh cong!")
    print(f"Database: {NEO4J_DATABASE}")
except Exception as e:
    print(f"LOI ket noi: {e}")
    raise

# ============================================
# 3. HIEN THI THONG KE TONG QUAN
# ============================================

print("\n[2] THONG KE TONG QUAN")
print("=" * 60)

with driver.session(database=NEO4J_DATABASE) as session:
    # Dem MedicalChunks
    result = session.run("MATCH (c:MedicalChunk) RETURN count(c) as count")
    total_chunks = result.single()['count']
    print(f"MedicalChunks: {total_chunks}")

    # Dem Entities
    result = session.run("MATCH (e:Entity) RETURN count(e) as count")
    total_entities = result.single()['count']
    print(f"Entities: {total_entities}")

    # Dem Relationships
    result = session.run("MATCH ()-[r]->() RETURN count(r) as count")
    total_relationships = result.single()['count']
    print(f"Relationships: {total_relationships}")

# ============================================
# 4. HIEN THI ENTITIES
# ============================================

print("\n[3] DANH SACH ENTITIES")
print("=" * 60)

with driver.session(database=NEO4J_DATABASE) as session:
    # Lay tat ca entities
    result = session.run("""
        MATCH (e:Entity)
        RETURN e.name AS name,
               e.type AS type,
               e.id AS id,
               e.source AS source,
               e.page AS page
        ORDER BY e.name
    """)

    entities = list(result)
    print(f"Tong so entities: {len(entities)}")

    # Hien thi bang
    df_entities = pd.DataFrame(entities)
    display(df_entities)

    # Thong ke theo loai
    print("\nThong ke theo loai entity:")
    type_counts = df_entities['type'].value_counts()
    for type_name, count in type_counts.items():
        print(f"  {type_name}: {count}")

# ============================================
# 5. HIEN THI RELATIONSHIPS
# ============================================

print("\n[4] DANH SACH RELATIONSHIPS")
print("=" * 60)

with driver.session(database=NEO4J_DATABASE) as session:
    # Lay tat ca relationships
    result = session.run("""
        MATCH (s:Entity)-[r]->(t:Entity)
        RETURN s.name AS source,
               type(r) AS type,
               t.name AS target,
               r.page AS page,
               r.source_chunk AS source_chunk
        ORDER BY s.name
    """)

    relationships = list(result)
    print(f"Tong so relationships: {len(relationships)}")

    if relationships:
        # Hien thi bang
        df_relationships = pd.DataFrame(relationships)
        display(df_relationships)

        # Thong ke theo loai relationship
        print("\nThong ke theo loai relationship:")
        type_counts = df_relationships['type'].value_counts()
        for type_name, count in type_counts.items():
            print(f"  {type_name}: {count}")
    else:
        print("Khong co relationships")

# ============================================
# 6. HIEN THI ENTITY VA RELATIONSHIP KET HOP
# ============================================

print("\n[5] ENTITY VA RELATIONSHIP KET HOP")
print("=" * 60)

with driver.session(database=NEO4J_DATABASE) as session:
    # Lay entities voi relationships cua chung
    result = session.run("""
        MATCH (e:Entity)
        OPTIONAL MATCH (e)-[r]->(t:Entity)
        RETURN e.name AS entity_name,
               e.type AS entity_type,
               collect(DISTINCT {target: t.name, type: type(r)}) AS out_relations,
               collect(DISTINCT {source: s.name, type: type(r2)}) AS in_relations
        OPTIONAL MATCH (s:Entity)-[r2]->(e)
        RETURN e.name AS entity_name,
               e.type AS entity_type,
               out_relations,
               in_relations
        LIMIT 20
    """)

    # Hien thi
    for record in result:
        entity = record['entity_name']
        entity_type = record['entity_type']
        out_rels = [r for r in record['out_relations'] if r.get('target')]
        in_rels = [r for r in record['in_relations'] if r.get('source')]

        print(f"\nEntity: {entity} ({entity_type})")

        if out_rels:
            print("  -> Relationships (di ra):")
            for r in out_rels[:5]:
                print(f"    -> {r['target']} ({r['type']})")

        if in_rels:
            print("  <- Relationships (di vao):")
            for r in in_rels[:5]:
                print(f"    <- {r['source']} ({r['type']})")

# ============================================
# 7. TIM KIEM ENTITY THEO TEN
# ============================================

print("\n[6] TIM KIEM ENTITY")
print("=" * 60)

def search_entities(driver, search_term):
    """Tim kiem entities theo ten"""

    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run("""
            MATCH (e:Entity)
            WHERE e.name CONTAINS $search_term
            RETURN e.name AS name,
                   e.type AS type,
                   e.id AS id,
                   e.page AS page
            ORDER BY e.name
            """,
            search_term=search_term
        )

        entities = list(result)
        print(f"Tim thay {len(entities)} entities chua '{search_term}':")

        if entities:
            df = pd.DataFrame(entities)
            display(df)
        else:
            print("Khong tim thay")

        return entities

# Tim kiem mau
search_terms = ["sot", "xuat huyet", "thuoc", "trieu"]
for term in search_terms:
    print(f"\n--- Tim kiem: {term} ---")
    search_entities(driver, term)

# ============================================
# 8. HIEN THI ENTITY VA CAC RELATIONSHIP CUA NO
# ============================================

print("\n[7] HIEN THI ENTITY VA CAC RELATIONSHIP CUA NO")
print("=" * 60)

def show_entity_relationships(driver, entity_name):
    """Hien thi entity va tat ca relationships cua no"""

    with driver.session(database=NEO4J_DATABASE) as session:
        # Kiem tra entity ton tai
        result = session.run("""
            MATCH (e:Entity {name: $entity_name})
            RETURN e.name AS name, e.type AS type
            """,
            entity_name=entity_name
        )

        entity = result.single()
        if not entity:
            print(f"Khong tim thay entity: {entity_name}")
            return

        print(f"\nEntity: {entity['name']} ({entity['type']})")
        print("-" * 40)

        # Lay relationships di ra
        result = session.run("""
            MATCH (e:Entity {name: $entity_name})-[r]->(t:Entity)
            RETURN t.name AS target, type(r) AS type, r.page AS page
            """,
            entity_name=entity_name
        )

        out_rels = list(result)
        print(f"\nRelationships di ra ({len(out_rels)}):")
        if out_rels:
            for r in out_rels:
                print(f"  -> {r['target']} --({r['type']})-- (page {r['page']})")
        else:
            print("  Khong co")

        # Lay relationships di vao
        result = session.run("""
            MATCH (s:Entity)-[r]->(e:Entity {name: $entity_name})
            RETURN s.name AS source, type(r) AS type, r.page AS page
            """,
            entity_name=entity_name
        )

        in_rels = list(result)
        print(f"\nRelationships di vao ({len(in_rels)}):")
        if in_rels:
            for r in in_rels:
                print(f"  <- {r['source']} --({r['type']})-- (page {r['page']})")
        else:
            print("  Khong co")

# Hien thi cho mot so entity
sample_entities = ["Sốt xuất huyết Dengue", "Xuất huyết", "Truyền dịch"]
for entity in sample_entities:
    show_entity_relationships(driver, entity)

# ============================================
# 9. EXPORT DU LIEU RA CSV
# ============================================

print("\n[8] EXPORT DU LIEU RA CSV")
print("=" * 40)

def export_to_csv(driver, output_dir="."):
    """Export entities va relationships ra CSV"""

    import pandas as pd

    with driver.session(database=NEO4J_DATABASE) as session:
        # Export entities
        result = session.run("""
            MATCH (e:Entity)
            RETURN e.name AS name,
                   e.type AS type,
                   e.id AS id,
                   e.source AS source,
                   e.page AS page
            ORDER BY e.name
        """)
        entities_df = pd.DataFrame(list(result))
        entities_df.to_csv(f"{output_dir}/entities.csv", index=False, encoding='utf-8-sig')
        print(f"Da luu entities vao {output_dir}/entities.csv")

        # Export relationships
        result = session.run("""
            MATCH (s:Entity)-[r]->(t:Entity)
            RETURN s.name AS source,
                   type(r) AS type,
                   t.name AS target,
                   r.page AS page,
                   r.source_chunk AS source_chunk
            ORDER BY s.name
        """)
        relationships_df = pd.DataFrame(list(result))
        relationships_df.to_csv(f"{output_dir}/relationships.csv", index=False, encoding='utf-8-sig')
        print(f"Da luu relationships vao {output_dir}/relationships.csv")

        return entities_df, relationships_df

# Export du lieu
entities_df, relationships_df = export_to_csv(driver)


print("\n[9] VISUALIZATION TRONG JUPYTER")
print("=" * 40)

def visualize_graph(driver, entity_name=None, max_entities=20):
    """Tao visualization cho graph trong Jupyter"""

    from IPython.display import HTML

    with driver.session(database=NEO4J_DATABASE) as session:
        if entity_name:
            # Lay entity va neighbors
            result = session.run("""
                MATCH (e:Entity {name: $entity_name})
                OPTIONAL MATCH (e)-[r]->(t:Entity)
                OPTIONAL MATCH (s:Entity)-[r2]->(e)
                RETURN e.name AS entity,
                       e.type AS entity_type,
                       collect(DISTINCT {name: t.name, type: t.type, rel: type(r)}) AS out_rels,
                       collect(DISTINCT {name: s.name, type: s.type, rel: type(r2)}) AS in_rels
                """,
                entity_name=entity_name
            )
            record = result.single()

            if record:
                print(f"\nGraph visualization cho entity: {entity_name}")
                print("-" * 40)

                # In entity
                print(f" {record['entity']} ({record['entity_type']})")

                # In out relationships
                out_rels = [r for r in record['out_rels'] if r.get('name')]
                if out_rels:
                    print("  └── Relationships di ra:")
                    for r in out_rels[:10]:
                        print(f"      └── {r['name']} ({r['type']}) --[{r['rel']}]-- ")

                # In in relationships
                in_rels = [r for r in record['in_rels'] if r.get('name')]
                if in_rels:
                    print("  └── Relationships di vao:")
                    for r in in_rels[:10]:
                        print(f"      └── {r['name']} ({r['type']}) --[{r['rel']}]-- ")
            else:
                print(f"Khong tim thay entity: {entity_name}")
        else:
            # Lay entities pho bien nhat
            result = session.run("""
                MATCH (e:Entity)
                OPTIONAL MATCH (e)-[r]->()
                WITH e, count(r) AS rel_count
                RETURN e.name AS name, e.type AS type, rel_count
                ORDER BY rel_count DESC
                LIMIT $max_entities
                """,
                max_entities=max_entities
            )

            print(f"\nEntities co nhieu relationships nhat ({max_entities} entities):")
            print("-" * 40)

            for record in result:
                bar = "" * min(record['rel_count'], 20)
                print(f"  {record['name']} ({record['type']}) - {record['rel_count']} relationships {bar}")

# Chay visualization
print("\nVISUALIZATION:")
visualize_graph(driver, "Sốt xuất huyết Dengue")
visualize_graph(driver, max_entities=15)

# ============================================
# 11. TONG KET
# ============================================

print("\n" + "=" * 60)
print("TONG KET DU LIEU TRONG NEO4J")
print("=" * 60)

with driver.session(database=NEO4J_DATABASE) as session:
    # Dem theo loai entity
    result = session.run("""
        MATCH (e:Entity)
        RETURN e.type AS type, count(e) AS count
        ORDER BY count DESC
    """)

    print("\nPhan bo Entity Types:")
    for record in result:
        bar = "" * min(record['count'], 20)
        print(f"  {record['type']}: {record['count']} {bar}")

    # Dem theo loai relationship
    result = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS type, count(r) AS count
        ORDER BY count DESC
    """)

    print("\nPhan bo Relationship Types:")
    for record in result:
        bar = "" * min(record['count'], 20)
        print(f"  {record['type']}: {record['count']} {bar}")

print("\n" + "=" * 60)
print("HOAN TAT!")
print("=" * 60)

# Dong ket noi
# driver.close()

HIEN THI ENTITIES VA RELATIONSHIPS TRONG NEO4J

[1] KET NOI NEO4J
----------------------------------------
Ket noi Neo4j thanh cong!
Database: 1db0cccb

[2] THONG KE TONG QUAN
MedicalChunks: 259
Entities: 55
Relationships: 53

[3] DANH SACH ENTITIES
Tong so entities: 55


,0,1,2,3,4
0,"0,5 - 1mg/kg",lieu_dung,f59e378375f095ef,unknown,28
1,10μg/kg/phút,lieu_dung,ceb6cf03ace66c37,unknown,28
2,AST/ALT,xet_nghiem,eeb556579de19115,unknown,14
3,"Bệnh lý tim, phổi, thận, mãn tính",trieu_chung,c1c61d530075947a,unknown,52
4,CPT,thuoc,84bc046e35897712,unknown,16
5,Chẩn đoán phân biệt nhiễm khuẩn huyết,trieu_chung,3b368c74a0f3093d,unknown,52
6,"Dextran 40, Dextran 70 hoặc 6% HES 200",thuoc,77740e6e66eb8a0a,unknown,16
7,Dextrose 30%,thuoc,cc4d9ea5a8c792fa,unknown,26
8,Diazepam,thuoc,820e0845fa3dd74c,unknown,26
9,Dobutamin,thuoc,73ba6e6af2f90ed5,unknown,54



Thong ke theo loai entity:


KeyError: 'type'

In [ ]:
# ============================================
# CELL: KIEM TRA RELATIONSHIPS
# ============================================

print("KIEM TRA RELATIONSHIPS TRONG NEO4J")
print("=" * 60)

with driver.session(database=NEO4J_DATABASE) as session:
    # 1. Dem so relationships
    result = session.run("MATCH ()-[r]->() RETURN count(r) as total")
    total = result.single()['total']
    print(f"Tong so relationships: {total}")

    if total == 0:
        print("\n⚠️ KHONG CO RELATIONSHIP NAO!")
        print("Co the do:")
        print("  1. Relationships chua duoc luu")
        print("  2. Entities khong co ket noi voi nhau")
        print("  3. Code luu relationships bi loi")
    else:
        # 2. Hien thi mau relationships
        print("\nMau relationships (10 cai dau):")
        result = session.run("""
            MATCH (s:Entity)-[r]->(t:Entity)
            RETURN s.name AS source,
                   type(r) AS type,
                   t.name AS target,
                   r.page AS page
            LIMIT 10
        """)

        for record in result:
            print(f"  {record['source']} --({record['type']})--> {record['target']} (page {record['page']})")

        # 3. Thong ke loai relationship
        print("\nThong ke loai relationship:")
        result = session.run("""
            MATCH ()-[r]->()
            RETURN type(r) AS type, count(r) AS count
            ORDER BY count DESC
        """)

        for record in result:
            print(f"  {record['type']}: {record['count']}")

KIEM TRA RELATIONSHIPS TRONG NEO4J
Tong so relationships: 53

Mau relationships (10 cai dau):
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Sốt cao (page 1)
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Hct (page 92)
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Sốt (page 22)
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Hematocrit giảm (page 22)
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Phù phổi cấp (page 22)
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Đau đầu (page 26)
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Xuất huyết (page 26)
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Sốc (page 52)
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Xuất huyết tiêu hóa (page 52)
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Tổn thương gan (page 52)

Thong ke loai relationship:
  DIEU_TRI: 27
  CO_TRIEU_CHUNG: 16
  XET_NGHIEM: 4
  CHI_DINH: 3
  TRIEU_CHUNG: 2
  MUC_DO: 1


In [ ]:


import os
import base64
import json
import re
import time
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from datetime import datetime
from collections import Counter
from functools import wraps
from tqdm import tqdm
from groq import Groq
from google.colab import userdata

# ============================================
# 1. RATE LIMITER VA RETRY
# ============================================

def retry_with_backoff(max_retries=5, base_delay=3):
    """Retry voi exponential backoff khi gap rate limit"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    error_msg = str(e)
                    if "429" in error_msg or "rate_limit" in error_msg.lower():
                        delay = base_delay * (2 ** attempt) + random.uniform(0, 2)
                        print(f"  Rate limit, thu lai sau {delay:.2f}s (lan {attempt+1}/{max_retries})")
                        time.sleep(delay)
                    elif "timeout" in error_msg.lower():
                        delay = base_delay + random.uniform(0, 1)
                        print(f"  Timeout, thu lai sau {delay:.2f}s")
                        time.sleep(delay)
                    else:
                        raise e
            raise Exception(f"Max retries exceeded: {func.__name__}")
        return wrapper
    return decorator

# ============================================
# 2. LAY DANH SACH ANH TU DRIVE
# ============================================

def get_images_from_drive(
    image_dir: Optional[str] = None,
    max_images: Optional[int] = 5
) -> List[Path]:
    """Lay danh sach anh tu Google Drive"""
    if image_dir is None:
        image_dir = "/content/drive/MyDrive/Images"

    image_dir = Path(image_dir)
    image_dir.mkdir(parents=True, exist_ok=True)

    extensions = ['.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp']
    images = []

    for ext in extensions:
        images.extend(image_dir.glob(f"*{ext}"))
        images.extend(image_dir.glob(f"*{ext.upper()}"))

    images = sorted(list(set(images)), key=lambda x: x.name)

    if max_images and len(images) > max_images:
        images = images[:max_images]

    print(f"Tim thay {len(images)} anh trong {image_dir}")
    return images

# ============================================
# 3. MA HOA ANH SANG BASE64
# ============================================

def encode_image(image_path: Path, max_size_mb: int = 20) -> Optional[str]:
    """Ma hoa anh sang base64"""
    try:
        if image_path.stat().st_size > max_size_mb * 1024 * 1024:
            print(f"  Anh qua lon (> {max_size_mb}MB): {image_path.name}")
            return None

        with open(image_path, "rb") as f:
            return base64.b64encode(f.read()).decode('utf-8')
    except Exception as e:
        print(f"  Loi doc anh {image_path.name}: {e}")
        return None

# ============================================
# 4. BUOC 1: GROQ VISION -> MO TA TEXT
# ============================================

@retry_with_backoff(max_retries=3, base_delay=3)
def extract_text_from_image_groq(
    image_path: Path,
    client: Groq,
    model: str = "qwen/qwen3.6-27b",
    temperature: float = 0.1,
    max_tokens: int = 1024  # Giam tokens de tranh rate limit
) -> str:
    """
    Buoc 1: Gui anh len Groq Vision de lay mo ta text
    """
    base64_img = encode_image(image_path)
    if not base64_img:
        return ""

    # Prompt ngan gon hon de tiet kiem tokens
    prompt = """Phan tich so do y te nay. Tra ve tieng Viet, mo ta:
1. Ten benh / so do
2. Cac nhom benh nhan
3. Muc do benh (nhe, vua, nang, canh bao)
4. Phac do dieu tri
5. Thuoc / dich truyen
6. Trieu chung / dau hieu

Chi tra ve mo ta, khong can JSON."""

    try:
        response = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/jpeg;base64,{base64_img}"}
                        }
                    ]
                }
            ],
            model=model,
            temperature=temperature,
            max_tokens=max_tokens
        )

        content = response.choices[0].message.content.strip()

        # Loai bo phan suy nghi (neu co)
        content = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL).strip()

        return content

    except Exception as e:
        print(f"  Loi Groq Vision: {e}")
        return ""

# ============================================
# 5. BUOC 2: LLAMA TEXT -> TRICH XUAT JSON
# ============================================

@retry_with_backoff(max_retries=3, base_delay=2)
def extract_json_from_text(
    text: str,
    client: Groq,
    model: str = "llama-3.1-8b-instant",
    temperature: float = 0.0
) -> Dict:
    """
    Buoc 2: Dung LLAMA de trich xuat entity tu text
    """
    if not text or len(text) < 20:
        return {"entities": [], "relationships": [], "description": ""}

    prompt = f"""Ban la chuyen gia y te. Phan tich van ban sau va tra ve JSON.

VAN BAN:
{text}

Tra ve JSON dung dinh dang:
{{
  "entities": [
    {{"name": "ten entity", "type": "disease|drug|symptom|dosage|treatment|patient_group|severity"}}
  ],
  "relationships": [
    {{"source": "ten nguon", "target": "ten dich", "type": "treats|has_symptom|has_dosage|has_risk"}}
  ],
  "description": "mo ta ngan gon"
}}

CHI TRA VE JSON, KHONG CO TEXT KHAC."""

    try:
        response = client.chat.completions.create(
            messages=[
                {"role": "system", "content": "Ban la chuyen gia y te. Chi tra ve JSON hop le."},
                {"role": "user", "content": prompt}
            ],
            model=model,
            temperature=temperature,
            max_tokens=1024
        )

        content = response.choices[0].message.content.strip()

        # Tim JSON trong content
        json_match = re.search(r'\{.*\}', content, re.DOTALL)
        if json_match:
            json_str = json_match.group()
            # Sua loi JSON
            json_str = re.sub(r',\s*}', '}', json_str)
            json_str = re.sub(r',\s*]', ']', json_str)
            json_str = re.sub(r'([{,])\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*:', r'\1"\2":', json_str)

            try:
                result = json.loads(json_str)
                if "entities" not in result:
                    result["entities"] = []
                if "relationships" not in result:
                    result["relationships"] = []
                if "description" not in result:
                    result["description"] = ""
                return result
            except json.JSONDecodeError:
                pass

        # Fallback: parse bang regex
        return parse_entities_with_regex_simple(content)

    except Exception as e:
        print(f"  Loi LLAMA: {e}")
        return {"entities": [], "relationships": [], "description": ""}

# ============================================
# 6. FALLBACK: PARSE BANG REGEX
# ============================================

def parse_entities_with_regex_simple(content: str) -> Dict:
    """Parse entity bang regex khi JSON khong hop le"""
    entities = []
    relationships = []

    # Tim entity patterns
    entity_pattern = r'["\']?name["\']?\s*[:=]\s*["\']([^"\']+)["\']?\s*,\s*["\']?type["\']?\s*[:=]\s*["\']([^"\']+)["\']?'
    matches = re.findall(entity_pattern, content)

    for name, etype in matches:
        entities.append({
            "name": name.strip(),
            "type": etype.strip().lower()
        })

    # Tim relationship patterns
    rel_pattern = r'["\']?source["\']?\s*[:=]\s*["\']([^"\']+)["\']?\s*,\s*["\']?target["\']?\s*[:=]\s*["\']([^"\']+)["\']?\s*,\s*["\']?type["\']?\s*[:=]\s*["\']([^"\']+)["\']?'
    matches = re.findall(rel_pattern, content)

    for source, target, reltype in matches:
        relationships.append({
            "source": source.strip(),
            "target": target.strip(),
            "type": reltype.strip()
        })

    return {
        "entities": entities,
        "relationships": relationships,
        "description": ""
    }

# ============================================
# 7. CHUAN HOA ENTITY
# ============================================

ENTITY_TYPE_MAPPING = {
    "disease": "disease",
    "drug": "drug",
    "symptom": "symptom",
    "dosage": "dosage",
    "treatment": "treatment",
    "indication": "indication",
    "contraindication": "contraindication",
    "vaccine": "vaccine",
    "test_result": "test_result",
    "patient": "patient",
    "patient_group": "patient_group",
    "severity": "severity",
    "food": "diet",
    "equipment": "medical_device",
    "device": "medical_device",
    "hospital": "location",
    "department": "location",
}

def normalize_entity_name(name: str) -> str:
    """Chuan hoa ten entity"""
    if not name:
        return ""

    name = name.strip()
    name = name.title()

    # Chuan hoa cac tu viet tat
    abbreviations = {
        "SXH": "Sot xuat huyet",
        "SXHD": "Sot xuat huyet Dengue",
        "Hct": "Hematocrit",
        "HA": "Huyet ap",
        "LS": "Lam sang",
        "CPT": "Cao phan tu",
        "RL": "Ringerlactate",
        "NaCl": "Natri Clorua",
        "ICU": "Hoi suc tich cuc",
        "DH": "Dau hieu",
    }

    for abbr, full in abbreviations.items():
        if name.upper() == abbr.upper():
            return full
        if abbr.upper() in name.upper():
            name = name.replace(abbr, full)

    return name

def normalize_entity_type(entity_type: str) -> str:
    """Chuan hoa loai entity"""
    entity_type = entity_type.lower().strip()
    return ENTITY_TYPE_MAPPING.get(entity_type, "unknown")

def normalize_entities(entities: List[Dict], source: str = "image") -> List[Dict]:
    """Chuan hoa danh sach entity"""
    normalized = []
    seen = set()

    for entity in entities:
        name = entity.get("name", "").strip()
        entity_type = entity.get("type", "unknown")

        if not name:
            continue

        norm_name = normalize_entity_name(name)
        norm_type = normalize_entity_type(entity_type)

        key = (norm_name.lower(), norm_type)
        if key not in seen:
            seen.add(key)
            normalized.append({
                "name": norm_name,
                "type": norm_type,
                "source": source,
                "original_name": name,
                "original_type": entity_type,
            })

    return normalized

# ============================================
# 8. XU LY ANH THEO BATCH (TOI UU RATE LIMIT)
# ============================================

def process_image_batch(
    images: List[Path],
    client: Groq,
    vision_model: str = "qwen/qwen3.6-27b",
    text_model: str = "llama-3.1-8b-instant",
    batch_size: int = 3,
    delay_between_images: float = 2.0,
    delay_between_batches: float = 10.0
) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    """
    Xu ly danh sach anh theo batch de tranh rate limit
    """
    all_entities = []
    all_relationships = []
    all_descriptions = []

    total_batches = (len(images) + batch_size - 1) // batch_size

    for batch_idx in range(total_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(images))
        batch = images[start_idx:end_idx]

        print(f"\nBatch {batch_idx + 1}/{total_batches}: Xu ly {len(batch)} anh")
        print("-" * 40)

        for img_idx, img in enumerate(batch):
            print(f"  [{img_idx + 1}/{len(batch)}] {img.name}")

            # Buoc 1: Vision -> text
            print(f"    Buoc 1: Phan tich anh...")
            description_text = extract_text_from_image_groq(
                img, client, vision_model
            )

            if not description_text:
                print(f"    Khong lay duoc mo ta")
                # Delay truoc khi tiep tuc
                time.sleep(delay_between_images)
                continue

            print(f"    Mo ta: {len(description_text)} ky tu")

            # Buoc 2: LLM -> JSON
            print(f"    Buoc 2: Trich xuat entity...")
            result = extract_json_from_text(
                description_text, client, text_model
            )

            entities = result.get("entities", [])
            relationships = result.get("relationships", [])
            description = result.get("description", "")

            if entities:
                print(f"    Tim thay {len(entities)} entities")
            else:
                print(f"    Khong co entity")

            # Chuan hoa entities
            normalized = normalize_entities(entities, "image_two_steps")

            for e in normalized:
                e["source_image"] = str(img)
                e["source_type"] = "image_two_steps"
                e["model"] = f"{vision_model}->{text_model}"
                all_entities.append(e)

            for r in relationships:
                r["source_image"] = str(img)
                r["source_type"] = "image_two_steps"
                r["model"] = f"{vision_model}->{text_model}"
                all_relationships.append(r)

            if description:
                all_descriptions.append({
                    "image": str(img),
                    "description": description,
                    "entities_count": len(entities),
                    "relationships_count": len(relationships)
                })

            # Delay giua cac anh trong batch
            if img_idx < len(batch) - 1:
                print(f"    Cho {delay_between_images}s truoc anh tiep theo...")
                time.sleep(delay_between_images)

        # Delay giua cac batch
        if batch_idx < total_batches - 1:
            print(f"\nCho {delay_between_batches}s truoc batch tiep theo...")
            time.sleep(delay_between_batches)

    return all_entities, all_relationships, all_descriptions

# ============================================
# 9. HAM CHINH: PROCESS ALL IMAGES
# ============================================

def process_all_images(
    image_dir: Optional[str] = None,
    max_images: Optional[int] = 5,
    vision_model: str = "qwen/qwen3.6-27b",
    text_model: str = "llama-3.1-8b-instant",
    batch_size: int = 3,
    save_to_file: bool = True,
    use_cache: bool = True
) -> Dict:
    """
    Xu ly tat ca anh trong thu muc

    Args:
        image_dir: Duong dan thu muc anh
        max_images: So anh toi da can xu ly (None = tat ca)
        vision_model: Model cho Groq Vision
        text_model: Model cho LLM trich xuat
        batch_size: So anh moi batch
        save_to_file: Luu ket qua ra file
        use_cache: Su dung cache neu co
    """

    # Lay API key
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except:
        GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

    if not GROQ_API_KEY:
        import getpass
        GROQ_API_KEY = getpass.getpass("Nhap GROQ_API_KEY: ")
        if not GROQ_API_KEY:
            print("Khong co GROQ_API_KEY")
            return {"entities": [], "relationships": [], "stats": {}}

    # Khoi tao Groq client
    try:
        client = Groq(api_key=GROQ_API_KEY)
        print("Da khoi tao Groq client")
    except Exception as e:
        print(f"Loi khoi tao Groq: {e}")
        return {"entities": [], "relationships": [], "stats": {}}

    # Lay danh sach anh
    images = get_images_from_drive(
        image_dir=image_dir,
        max_images=max_images
    )

    if not images:
        print("Khong co anh nao de xu ly")
        return {"entities": [], "relationships": [], "stats": {"total_images": 0}}

    # Kiem tra cache
    cache_file = Path("image_processed_cache.json")
    if use_cache and cache_file.exists():
        try:
            with open(cache_file, 'r', encoding='utf-8') as f:
                cached_data = json.load(f)
            print(f"Da load cache tu {cache_file}")
            cached_images = set(cached_data.get('processed_images', []))
            # Chi xu ly anh chua co trong cache
            new_images = [img for img in images if str(img) not in cached_images]
            if not new_images:
                print("Tat ca anh da duoc xu ly, load tu cache")
                return {
                    "entities": cached_data.get('entities', []),
                    "relationships": cached_data.get('relationships', []),
                    "descriptions": cached_data.get('descriptions', []),
                    "stats": cached_data.get('stats', {})
                }
            images = new_images
            print(f"Xu ly {len(images)} anh moi (da cache {len(cached_images)} anh)")
        except Exception as e:
            print(f"Loi doc cache: {e}")

    print(f"\nXU LY {len(images)} ANH QUA 2 BUOC")
    print(f"   Buoc 1: {vision_model} -> mo ta text")
    print(f"   Buoc 2: {text_model} -> trich xuat JSON")
    print(f"   Batch size: {batch_size}")
    print("=" * 60)

    # Xu ly anh theo batch
    all_entities, all_relationships, all_descriptions = process_image_batch(
        images=images,
        client=client,
        vision_model=vision_model,
        text_model=text_model,
        batch_size=batch_size
    )

    # Thong ke
    print("\n" + "=" * 60)
    print("THONG KE XU LY ANH")
    print("=" * 60)
    print(f"   Tong anh: {len(images)}")
    print(f"   Tong entities (sau chuan hoa): {len(all_entities)}")
    print(f"   Tong relationships: {len(all_relationships)}")
    print(f"   Mo ta anh: {len(all_descriptions)}")

    if all_entities:
        entity_types = Counter([e.get('type', 'unknown') for e in all_entities])
        print("\nPHAN BO ENTITY TYPES:")
        for t, c in entity_types.most_common(5):
            print(f"   {t}: {c}")

    result = {
        "entities": all_entities,
        "relationships": all_relationships,
        "descriptions": all_descriptions,
        "stats": {
            "total_images": len(images),
            "total_entities": len(all_entities),
            "total_relationships": len(all_relationships),
            "vision_model": vision_model,
            "text_model": text_model,
            "mode": "two_steps",
            "timestamp": datetime.now().isoformat()
        },
        "processed_images": [str(img) for img in images]
    }

    # Luu ket qua
    if save_to_file:
        with open('image_entities.json', 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print("\nDa luu vao image_entities.json")

        # Luu cache
        if use_cache:
            # Merge voi cache cu
            if cache_file.exists():
                try:
                    with open(cache_file, 'r', encoding='utf-8') as f:
                        old_cache = json.load(f)
                    old_cache['entities'].extend(result['entities'])
                    old_cache['relationships'].extend(result['relationships'])
                    old_cache['descriptions'].extend(result['descriptions'])
                    old_cache['processed_images'].extend(result['processed_images'])
                    old_cache['stats']['total_images'] += result['stats']['total_images']
                    old_cache['stats']['total_entities'] += result['stats']['total_entities']
                    old_cache['stats']['total_relationships'] += result['stats']['total_relationships']
                    result = old_cache
                except Exception:
                    pass

            with open(cache_file, 'w', encoding='utf-8') as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            print("Da cap nhat cache")

    return result

# ============================================
# 10. TEST VOI 1 ANH
# ============================================

def test_single_image(image_path: Path):
    """Test xu ly 1 anh duy nhat va in ket qua chi tiet"""

    # Lay API key
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except:
        GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

    if not GROQ_API_KEY:
        import getpass
        GROQ_API_KEY = getpass.getpass("Nhap GROQ_API_KEY: ")

    client = Groq(api_key=GROQ_API_KEY)

    print(f"\nTEST XU LY ANH: {image_path.name}")
    print("=" * 60)

    # Buoc 1: Vision
    print("Buoc 1: Groq Vision -> Mo ta text")
    description = extract_text_from_image_groq(image_path, client)

    if not description:
        print("Khong lay duoc mo ta")
        return

    print(f"\nMO TA TEXT:\n{description[:500]}...")
    print(f"\nDo dai: {len(description)} ky tu")

    # Buoc 2: LLM -> JSON
    print("\nBuoc 2: LLAMA -> Trich xuat JSON")
    result = extract_json_from_text(description, client)

    print("\nKET QUA:")
    print(f"  Entities: {len(result.get('entities', []))}")
    for e in result.get('entities', [])[:5]:
        print(f"    - {e.get('name')} ({e.get('type')})")

    print(f"\n  Relationships: {len(result.get('relationships', []))}")
    for r in result.get('relationships', [])[:5]:
        print(f"    - {r.get('source')} --({r.get('type')})--> {r.get('target')}")

    print(f"\n  Description: {result.get('description', '')[:200]}...")

    return description, result

# ============================================
# 11. CHAY CHINH
# ============================================

print("CELL 12: XU LY ANH QUA 2 BUOC")
print("=" * 60)

# Kiem tra ket noi
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=GROQ_API_KEY)
    print("Ket noi Groq thanh cong")
except Exception as e:
    print(f"Loi ket noi Groq: {e}")
    print("Khong the xu ly anh")
    image_result = {"entities": [], "relationships": [], "stats": {}}
    client = None

if client:
    # TEST: Xu ly 1 anh truoc de kiem tra
    print("\n" + "=" * 60)
    print("TEST VOI 1 ANH TRUOC")
    print("=" * 60)

    test_images = get_images_from_drive(max_images=1)
    if test_images:
        test_single_image(test_images[0])

    # XU LY TAT CA ANH
    print("\n" + "=" * 60)
    print("XU LY TAT CA ANH")
    print("=" * 60)

    image_result = process_all_images(
        image_dir="/content/drive/MyDrive/Images",
        max_images=5,  # Gioi han 5 anh de tranh rate limit
        vision_model="qwen/qwen3.6-27b",
        text_model="llama-3.1-8b-instant",
        batch_size=2,  # Batch nho de tranh rate limit
        save_to_file=True,
        use_cache=True
    )

# Gop voi text data
if 'extracted_data' in globals() and extracted_data and image_result:
    print("\n" + "=" * 60)
    print("GOP TEXT + IMAGE")
    print("=" * 60)

    merged_data = {
        "entities": extracted_data.get("entities", []) + image_result.get("entities", []),
        "relationships": extracted_data.get("relationships", []) + image_result.get("relationships", []),
    }

    # Loai bo trung lap
    seen = set()
    unique_entities = []
    for e in merged_data["entities"]:
        key = (e.get('name', '').lower(), e.get('type', '').lower())
        if key not in seen:
            seen.add(key)
            unique_entities.append(e)

    seen = set()
    unique_relationships = []
    for r in merged_data["relationships"]:
        key = (
            r.get('source', '').lower(),
            r.get('target', '').lower(),
            r.get('type', '').lower()
        )
        if key not in seen:
            seen.add(key)
            unique_relationships.append(r)

    merged_data["entities"] = unique_entities
    merged_data["relationships"] = unique_relationships

    print(f"   Text entities: {len(extracted_data.get('entities', []))}")
    print(f"   Image entities: {len(image_result.get('entities', []))}")
    print(f"   Tong entities (sau gop): {len(unique_entities)}")
    print(f"   Text relationships: {len(extracted_data.get('relationships', []))}")
    print(f"   Image relationships: {len(image_result.get('relationships', []))}")
    print(f"   Tong relationships (sau gop): {len(unique_relationships)}")

    # Luu merged data
    with open('merged_entities.json', 'w', encoding='utf-8') as f:
        json.dump(merged_data, f, ensure_ascii=False, indent=2)
    print("Da luu merged_entities.json")


CELL 12: XU LY ANH QUA 2 BUOC
Ket noi Groq thanh cong

TEST VOI 1 ANH TRUOC
Tim thay 1 anh trong /content/drive/MyDrive/Images

TEST XU LY ANH: Screenshot 2026-08-13 035020.png
Buoc 1: Groq Vision -> Mo ta text

MO TA TEXT:
<think>
The user wants me to analyze a medical flowchart image and extract specific information in Vietnamese.

**1. Identify the Chart/Disease Name:**
*   Looking at the top of the image, the title is clearly visible: "PHỤ LỤC 3: SƠ ĐỒ PHÂN NHÓM ĐIỀU TRỊ NGƯỜI BỆNH SỐT XUẤT HUYẾT DENGUE".
*   So, the disease is "Sốt xuất huyết Dengue" (Dengue Hemorrhagic Fever).

**2. Identify Patient Groups:**
*   I need to look for boxes that list specific conditions or demographics.
*   There is a box on the...

Do dai: 3564 ky tu

Buoc 2: LLAMA -> Trich xuat JSON

KET QUA:
  Entities: 26
    - Sốt xuất huyết Dengue (disease)
    - PHỤ LỤC 3: SƠ ĐỒ PHÂN NHÓM ĐIỀU TRỊ NGƯỜI BỆNH SỐT XUẤT HUYẾT DENGUE (chart)
    - Sống một mình (patient_group)
    - Nhà quá xa cơ sở y tế... (pati

In [ ]:
# ============================================
# CELL 13.1: TAO SCHEMA CHO NEO4J
# ============================================

def create_schema(driver):
    """Tao schema va constraints cho Neo4j"""

    with driver.session(database=NEO4J_DATABASE) as session:
        # 1. Constraints cho MedicalChunk
        session.run("""
            CREATE CONSTRAINT medical_chunk_id_unique
            IF NOT EXISTS
            FOR (c:MedicalChunk) REQUIRE c.chunk_id IS UNIQUE
        """)
        print("Da tao constraint cho MedicalChunk")

        # 2. Constraints cho Entity
        session.run("""
            CREATE CONSTRAINT entity_id_unique
            IF NOT EXISTS
            FOR (e:Entity) REQUIRE e.id IS UNIQUE
        """)
        print("Da tao constraint cho Entity")

        # 3. Constraints cho Image
        session.run("""
            CREATE CONSTRAINT image_path_unique
            IF NOT EXISTS
            FOR (i:Image) REQUIRE i.path IS UNIQUE
        """)
        print("Da tao constraint cho Image")

        # 4. Index cho text search
        session.run("""
            CREATE FULLTEXT INDEX chunk_text_index
            IF NOT EXISTS
            FOR (c:MedicalChunk)
            ON EACH [c.text]
        """)
        print("Da tao fulltext index cho MedicalChunk")

        # 5. Vector index cho entity embeddings
        session.run("""
            CREATE VECTOR INDEX entity_embeddings IF NOT EXISTS
            FOR (e:Entity) ON (e.embedding)
            OPTIONS {indexConfig: {
                `vector.dimensions`: 768,
                `vector.similarity_function`: 'cosine'
            }}
        """)
        print("Da tao vector index cho Entity")

        # 6. Index cho page
        session.run("""
            CREATE INDEX medical_chunk_page_idx
            IF NOT EXISTS
            FOR (c:MedicalChunk) ON (c.page)
        """)
        print("Da tao index cho page")

        print("\nDa tao toan bo schema cho Neo4j")

# Khoi tao driver va tao schema
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

create_schema(driver)

Da tao constraint cho MedicalChunk
Da tao constraint cho Entity
Da tao constraint cho Image
Da tao fulltext index cho MedicalChunk
Da tao vector index cho Entity
Da tao index cho page

Da tao toan bo schema cho Neo4j


In [ ]:
# ============================================
# CELL 13.2: LUU CHUNKS VOI RETRY VA DELAY
# ============================================

import time
import random
from functools import wraps
from neo4j import GraphDatabase
import hashlib
import json

def retry_on_quota(max_retries=10, base_delay=5):
    """Retry khi gap quota exhausted"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    error_msg = str(e)
                    # Kiem tra neu la quota error
                    if "RESOURCE_EXHAUSTED" in error_msg or "429" in error_msg:
                        # Tinh delay exponential backoff
                        delay = base_delay * (2 ** attempt) + random.uniform(0, 5)
                        print(f"  Quota exhausted, thu lai sau {delay:.2f}s (lan {attempt+1}/{max_retries})")
                        time.sleep(delay)
                    elif "timeout" in error_msg.lower():
                        delay = base_delay + random.uniform(0, 2)
                        print(f"  Timeout, thu lai sau {delay:.2f}s")
                        time.sleep(delay)
                    else:
                        # Loi khac, raise len
                        raise e
            raise Exception(f"Max retries exceeded: {func.__name__}")
        return wrapper
    return decorator

def save_chunks_to_neo4j_safe(driver, chunks, embeddings, batch_size=20):
    """
    Luu chunks vao Neo4j voi retry va delay de tranh quota
    """

    print(f"Bat dau luu {len(chunks)} chunks vao Neo4j...")
    print("=" * 60)
    print(f"Batch size: {batch_size} (giam de tranh quota)")

    # Chi luu cac chunks chua co trong database
    with driver.session(database=NEO4J_DATABASE) as session:
        # Lay cac chunk_id da co
        result = session.run("MATCH (c:MedicalChunk) RETURN c.chunk_id as chunk_id")
        existing_ids = set(record['chunk_id'] for record in result)
        print(f"Da co {len(existing_ids)} chunks trong database")

        # Loc chunks chua luu
        chunks_to_save = []
        for chunk in chunks:
            chunk_id = hashlib.md5(chunk.page_content.encode('utf-8')).hexdigest()[:16]
            if chunk_id not in existing_ids:
                chunks_to_save.append((chunk_id, chunk))

        print(f"Can luu them {len(chunks_to_save)} chunks")

        if not chunks_to_save:
            print("Tat ca chunks da duoc luu!")
            return

    total_batches = (len(chunks_to_save) + batch_size - 1) // batch_size

    with driver.session(database=NEO4J_DATABASE) as session:
        for batch_idx in range(total_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(chunks_to_save))
            batch = chunks_to_save[start_idx:end_idx]

            print(f"\nBatch {batch_idx + 1}/{total_batches}: Luu {len(batch)} chunks")
            print("-" * 40)

            for i, (chunk_id, chunk) in enumerate(batch):
                # Retry cho tung chunk
                saved = False
                for attempt in range(5):
                    try:
                        # Tao embedding voi retry
                        embedding = embeddings.embed_query(chunk.page_content)

                        # Luu vao Neo4j
                        session.run("""
                            MERGE (c:MedicalChunk {chunk_id: $chunk_id})
                            SET c.text = $text,
                                c.page = $page,
                                c.type = $type,
                                c.embedding = $embedding,
                                c.metadata = $metadata,
                                c.created_at = datetime(),
                                c.updated_at = datetime()
                            """,
                            chunk_id=chunk_id,
                            text=chunk.page_content,
                            page=chunk.metadata.get('page', 0),
                            type=chunk.metadata.get('type', 'text'),
                            embedding=embedding,
                            metadata=json.dumps(chunk.metadata)
                        )

                        saved = True
                        break

                    except Exception as e:
                        if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                            delay = 5 * (2 ** attempt) + random.uniform(0, 3)
                            print(f"    Quota exhausted, thu lai sau {delay:.2f}s")
                            time.sleep(delay)
                        else:
                            print(f"    Loi: {e}")
                            break

                if saved:
                    if (i + 1) % 5 == 0:
                        print(f"  Da luu {i + 1}/{len(batch)} chunks trong batch")
                else:
                    print(f"  Khong the luu chunk {chunk_id}")

                # Delay nho giua cac chunks
                time.sleep(0.5)

            print(f"  Hoan thanh batch {batch_idx + 1}")

            # Delay lon giua cac batch
            if batch_idx < total_batches - 1:
                print(f"\nCho 30s truoc batch tiep theo...")
                time.sleep(30)

    print(f"\nDa luu {len(chunks_to_save)} chunks vao Neo4j")

    # Kiem tra so luong da luu
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run("MATCH (c:MedicalChunk) RETURN count(c) as total")
        total = result.single()['total']
        print(f"Tong so chunks trong Neo4j: {total}")

# CHAY VOI BATCH SIZE NHO
if 'final_chunks' in globals() and final_chunks:
    print(f"Co {len(final_chunks)} chunks de luu")

    # Luu chunks vao Neo4j voi batch size nho
    save_chunks_to_neo4j_safe(
        driver=driver,
        chunks=final_chunks,
        embeddings=embeddings,
        batch_size=10  # Giam batch size de tranh quota
    )
else:
    print("Khong co chunks de luu")

Co 260 chunks de luu
Bat dau luu 260 chunks vao Neo4j...
Batch size: 10 (giam de tranh quota)
Da co 196 chunks trong database
Can luu them 63 chunks

Batch 1/7: Luu 10 chunks
----------------------------------------
  Da luu 5/10 chunks trong batch
  Da luu 10/10 chunks trong batch
  Hoan thanh batch 1

Cho 30s truoc batch tiep theo...

Batch 2/7: Luu 10 chunks
----------------------------------------
  Da luu 5/10 chunks trong batch
  Da luu 10/10 chunks trong batch
  Hoan thanh batch 2

Cho 30s truoc batch tiep theo...

Batch 3/7: Luu 10 chunks
----------------------------------------
  Da luu 5/10 chunks trong batch
  Da luu 10/10 chunks trong batch
  Hoan thanh batch 3

Cho 30s truoc batch tiep theo...

Batch 4/7: Luu 10 chunks
----------------------------------------
  Da luu 5/10 chunks trong batch
  Da luu 10/10 chunks trong batch
  Hoan thanh batch 4

Cho 30s truoc batch tiep theo...

Batch 5/7: Luu 10 chunks
----------------------------------------
  Da luu 5/10 chunks trong b

In [ ]:
# ============================================
# CELL 13.3: LUU ENTITIES VAO NEO4J
# ============================================

def save_entities_to_neo4j(driver, entities, batch_size=50):
    """
    Luu entities vao Neo4j

    Args:
        driver: Neo4j driver
        entities: List of entity dictionaries
        batch_size: So entities luu moi batch
    """

    if not entities:
        print("Khong co entities de luu")
        return

    print(f"\nBat dau luu {len(entities)} entities vao Neo4j...")
    print("=" * 60)

    # Loai bo entities trung lap truoc khi luu
    seen = set()
    unique_entities = []
    for e in entities:
        key = (e.get('name', '').strip().lower(), e.get('type', '').strip().lower())
        if key not in seen:
            seen.add(key)
            unique_entities.append(e)

    print(f"Loai bo trung lap: {len(entities)} -> {len(unique_entities)} entities")

    total_batches = (len(unique_entities) + batch_size - 1) // batch_size

    with driver.session(database=NEO4J_DATABASE) as session:
        for batch_idx in range(total_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(unique_entities))
            batch_entities = unique_entities[start_idx:end_idx]

            print(f"\nBatch {batch_idx + 1}/{total_batches}: Luu {len(batch_entities)} entities")

            for i, entity in enumerate(batch_entities):
                try:
                    name = entity.get('name', '').strip()
                    entity_type = entity.get('type', 'unknown')

                    if not name:
                        continue

                    # Tao entity_id
                    entity_id = hashlib.md5(
                        f"{name}_{entity_type}".encode('utf-8')
                    ).hexdigest()[:16]

                    # Luu vao Neo4j
                    session.run("""
                        MERGE (e:Entity {id: $entity_id})
                        SET e.name = $name,
                            e.type = $type,
                            e.source = $source,
                            e.source_type = $source_type,
                            e.page = $page,
                            e.model = $model,
                            e.created_at = datetime(),
                            e.updated_at = datetime()
                        """,
                        entity_id=entity_id,
                        name=name,
                        type=entity_type,
                        source=entity.get('source', 'unknown'),
                        source_type=entity.get('source_type', 'text'),
                        page=entity.get('page', 0),
                        model=entity.get('model', 'unknown')
                    )

                except Exception as e:
                    print(f"  Loi luu entity {name}: {e}")
                    continue

            print(f"  Hoan thanh batch {batch_idx + 1}")

    print(f"\nDa luu {len(unique_entities)} entities vao Neo4j")

    # Kiem tra so luong da luu
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run("""
            MATCH (e:Entity)
            RETURN count(e) as total,
                   collect(distinct e.type) as types
        """)
        record = result.single()
        print(f"Tong so entities trong Neo4j: {record['total']}")
        print(f"Cac loai entity: {', '.join(record['types'][:10])}...")

# Kiem tra xem da co extracted_data chua
if 'extracted_data' in globals() and extracted_data:
    entities = extracted_data.get('entities', [])
    print(f"Co {len(entities)} entities de luu")

    # Luu entities vao Neo4j
    save_entities_to_neo4j(
        driver=driver,
        entities=entities,
        batch_size=100
    )
else:
    print("Khong co entities de luu. Hay chay Cell 11 truoc.")

Khong co entities de luu. Hay chay Cell 11 truoc.


In [ ]:
# ============================================
# CELL 13.3: LUU ENTITIES VAO NEO4J
# ============================================

def save_entities_to_neo4j(driver, entities, batch_size=100):
    """
    Luu entities vao Neo4j

    Args:
        driver: Neo4j driver
        entities: List of entity dictionaries
        batch_size: So entities luu moi batch
    """

    if not entities:
        print("Khong co entities de luu")
        return

    print(f"\nBat dau luu {len(entities)} entities vao Neo4j...")
    print("=" * 60)

    # Loai bo entities trung lap truoc khi luu
    seen = set()
    unique_entities = []
    for e in entities:
        key = (e.get('name', '').strip().lower(), e.get('type', '').strip().lower())
        if key not in seen:
            seen.add(key)
            unique_entities.append(e)

    print(f"Loai bo trung lap: {len(entities)} -> {len(unique_entities)} entities")

    total_batches = (len(unique_entities) + batch_size - 1) // batch_size

    with driver.session(database=NEO4J_DATABASE) as session:
        for batch_idx in range(total_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(unique_entities))
            batch_entities = unique_entities[start_idx:end_idx]

            print(f"\nBatch {batch_idx + 1}/{total_batches}: Luu {len(batch_entities)} entities")

            for i, entity in enumerate(batch_entities):
                try:
                    name = entity.get('name', '').strip()
                    entity_type = entity.get('type', 'unknown')

                    if not name:
                        continue

                    # Tao entity_id
                    entity_id = hashlib.md5(
                        f"{name}_{entity_type}".encode('utf-8')
                    ).hexdigest()[:16]

                    # Luu vao Neo4j
                    session.run("""
                        MERGE (e:Entity {id: $entity_id})
                        SET e.name = $name,
                            e.type = $type,
                            e.source = $source,
                            e.source_type = $source_type,
                            e.page = $page,
                            e.model = $model,
                            e.created_at = datetime(),
                            e.updated_at = datetime()
                        """,
                        entity_id=entity_id,
                        name=name,
                        type=entity_type,
                        source=entity.get('source', 'unknown'),
                        source_type=entity.get('source_type', 'text'),
                        page=entity.get('page', 0),
                        model=entity.get('model', 'unknown')
                    )

                except Exception as e:
                    print(f"  Loi luu entity {name}: {e}")
                    continue

            print(f"  Hoan thanh batch {batch_idx + 1}")

    print(f"\nDa luu {len(unique_entities)} entities vao Neo4j")

    # Kiem tra so luong da luu
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run("""
            MATCH (e:Entity)
            RETURN count(e) as total,
                   collect(distinct e.type) as types
        """)
        record = result.single()
        print(f"Tong so entities trong Neo4j: {record['total']}")
        print(f"Cac loai entity: {', '.join(record['types'][:10])}...")

# Kiem tra xem da co extracted_data chua
if 'extracted_data' in globals() and extracted_data:
    entities = extracted_data.get('entities', [])
    print(f"Co {len(entities)} entities de luu")

    # Luu entities vao Neo4j
    save_entities_to_neo4j(
        driver=driver,
        entities=entities,
        batch_size=100
    )
else:
    print("Khong co entities de luu. Hay chay Cell 11 truoc.")

Co 55 entities de luu

Bat dau luu 55 entities vao Neo4j...
Loai bo trung lap: 55 -> 55 entities

Batch 1/1: Luu 55 entities
  Hoan thanh batch 1

Da luu 55 entities vao Neo4j
Tong so entities trong Neo4j: 55
Cac loai entity: benh, thuoc, trieu_chung, dich_truyen, xet_nghiem, dieu_tri, muc_do, lieu_dung, nhom_benh_nhan, dụng_cụ...


In [ ]:
# ============================================
# CELL 13.4: LUU RELATIONSHIPS (DA SUA CHO PHU HOP)
# ============================================

import hashlib
import time
from neo4j import GraphDatabase

def save_relationships_final(driver, relationships, batch_size=50):
    """
    Luu relationships vao Neo4j - DA SUA CHO PHU HOP VOI ENTITY TYPES
    """

    if not relationships:
        print("Khong co relationships de luu")
        return

    print(f"\nBat dau luu {len(relationships)} relationships vao Neo4j...")
    print("=" * 60)

    # Loai bo trung lap
    seen = set()
    unique_relationships = []
    for r in relationships:
        source = r.get('source', '').strip()
        target = r.get('target', '').strip()
        rel_type = r.get('type', 'treats').strip().lower()

        if not source or not target:
            continue

        key = (source.lower(), target.lower(), rel_type.lower())
        if key not in seen:
            seen.add(key)
            unique_relationships.append(r)

    print(f"Loai bo trung lap: {len(relationships)} -> {len(unique_relationships)} relationships")

    if not unique_relationships:
        print("Khong co relationships hop le")
        return

    # Lay danh sach entity ids va names tu Neo4j
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run("""
            MATCH (e:Entity)
            RETURN e.id as id, e.name as name, e.type as type
        """)

        # Tao map voi ca ten va id
        entity_by_name = {}
        entity_by_id = {}
        for record in result:
            name = record['name'].lower().strip()
            entity_by_name[name] = {
                'id': record['id'],
                'type': record['type']
            }
            entity_by_id[record['id']] = record['name']

        print(f"Tim thay {len(entity_by_name)} entities trong Neo4j")

        # Kiem tra relationships co the tao
        valid_relationships = []
        invalid_count = 0
        missing_entities = set()

        for r in unique_relationships:
            source = r.get('source', '').strip()
            target = r.get('target', '').strip()
            rel_type = r.get('type', 'treats').strip().lower()

            # Chuan hoa relationship type
            rel_type_map = {
                'treats': 'DIEU_TRI',
                'has_symptom': 'CO_TRIEU_CHUNG',
                'has_dosage': 'CO_LIEU_DUNG',
                'contraindicated': 'CHONG_CHI_DINH',
                'prevents': 'PHONG_NGUA',
                'diagnoses': 'CHAN_DOAN',
                'interacts_with': 'TUONG_TAC',
                'causes': 'GAY_RA',
                'indicates': 'CHI_DINH',
            }
            rel_type_upper = rel_type_map.get(rel_type.lower(), rel_type.upper())

            source_info = entity_by_name.get(source.lower())
            target_info = entity_by_name.get(target.lower())

            if source_info and target_info:
                valid_relationships.append({
                    'source_id': source_info['id'],
                    'target_id': target_info['id'],
                    'source_name': source,
                    'target_name': target,
                    'type': rel_type_upper,
                    'page': r.get('page', 0),
                    'source_chunk': r.get('source_chunk', 'unknown'),
                    'source_type': r.get('source_type', 'text'),
                    'model': r.get('model', 'unknown')
                })
            else:
                invalid_count += 1
                if not source_info:
                    missing_entities.add(source)
                if not target_info:
                    missing_entities.add(target)

        print(f"\nRelationships hop le: {len(valid_relationships)}")
        print(f"Relationships khong hop le: {invalid_count}")

        if missing_entities:
            print(f"\nCac entity khong tim thay (can luu them):")
            for name in list(missing_entities)[:10]:
                print(f"  - {name}")

        if not valid_relationships:
            print("Khong co relationships hop le de luu!")
            return

    # Luu relationships
    total_batches = (len(valid_relationships) + batch_size - 1) // batch_size

    with driver.session(database=NEO4J_DATABASE) as session:
        for batch_idx in range(total_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(valid_relationships))
            batch = valid_relationships[start_idx:end_idx]

            print(f"\nBatch {batch_idx + 1}/{total_batches}: Luu {len(batch)} relationships")
            print("-" * 40)

            for i, rel in enumerate(batch):
                try:
                    # Tao relationship voi type da chuan hoa
                    query = f"""
                        MATCH (s:Entity {{id: $source_id}})
                        MATCH (t:Entity {{id: $target_id}})
                        MERGE (s)-[r:{rel['type']}]->(t)
                        SET r.page = $page,
                            r.source_chunk = $source_chunk,
                            r.source_type = $source_type,
                            r.model = $model,
                            r.created_at = datetime()
                    """

                    result = session.run(
                        query,
                        source_id=rel['source_id'],
                        target_id=rel['target_id'],
                        page=rel['page'],
                        source_chunk=rel['source_chunk'],
                        source_type=rel['source_type'],
                        model=rel['model']
                    )

                    # Kiem tra ket qua
                    summary = result.consume()
                    if summary.counters.contains_updates:
                        if (i + 1) % 5 == 0:
                            print(f"  Da luu {i + 1}/{len(batch)} relationships")
                    else:
                        print(f"  Khong tao duoc relationship: {rel['source_name']} -> {rel['target_name']}")

                except Exception as e:
                    print(f"  Loi luu relationship {rel['source_name']}->{rel['target_name']}: {e}")
                    continue

                # Delay nho
                time.sleep(0.1)

            print(f"  Hoan thanh batch {batch_idx + 1}")
            if batch_idx < total_batches - 1:
                print(f"Cho 3s truoc batch tiep theo...")
                time.sleep(3)

    print(f"\nDa luu {len(valid_relationships)} relationships vao Neo4j")

    # Kiem tra ket qua
    with driver.session(database=NEO4J_DATABASE) as session:
        # Dem relationships
        result = session.run("""
            MATCH ()-[r]->()
            RETURN count(r) as total,
                   collect(distinct type(r)) as types
        """)
        record = result.single()
        print(f"\nTong so relationships trong Neo4j: {record['total']}")
        print(f"Cac loai relationship: {', '.join(record['types'][:10])}")

        # Mau relationships
        result = session.run("""
            MATCH (s:Entity)-[r]->(t:Entity)
            RETURN s.name as source, type(r) as type, t.name as target
            LIMIT 10
        """)
        print("\nMau relationships:")
        for record in result:
            print(f"  {record['source']} --({record['type']})--> {record['target']}")

# ============================================
# CHAY LUU RELATIONSHIPS
# ============================================

if 'extracted_data' in globals() and extracted_data:
    relationships = extracted_data.get('relationships', [])
    print(f"Co {len(relationships)} relationships de luu")

    if relationships:
        save_relationships_final(
            driver=driver,
            relationships=relationships,
            batch_size=50
        )
    else:
        print("Khong co relationships de luu")
else:
    print("Khong co extracted_data. Hay chay Cell 11 truoc.")

Co 54 relationships de luu

Bat dau luu 54 relationships vao Neo4j...
Loai bo trung lap: 54 -> 53 relationships
Tim thay 53 entities trong Neo4j

Relationships hop le: 53
Relationships khong hop le: 0

Batch 1/2: Luu 50 relationships
----------------------------------------
  Da luu 5/50 relationships
  Da luu 10/50 relationships
  Da luu 15/50 relationships
  Da luu 20/50 relationships
  Da luu 25/50 relationships
  Da luu 30/50 relationships
  Da luu 35/50 relationships
  Da luu 40/50 relationships
  Da luu 45/50 relationships
  Da luu 50/50 relationships
  Hoan thanh batch 1
Cho 3s truoc batch tiep theo...

Batch 2/2: Luu 3 relationships
----------------------------------------
  Hoan thanh batch 2

Da luu 53 relationships vao Neo4j

Tong so relationships trong Neo4j: 53
Cac loai relationship: CO_TRIEU_CHUNG, DIEU_TRI, CHI_DINH, XET_NGHIEM, TRIEU_CHUNG, MUC_DO

Mau relationships:
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Sốt cao
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Hct


In [ ]:
# ============================================
# CELL 14.0: KIEM TRA DU LIEU TRONG NEO4J
# ============================================

def verify_data(driver):
    """Kiem tra tong quan du lieu da luu"""

    print("=" * 60)
    print("KIEM TRA DU LIEU TRONG NEO4J")
    print("=" * 60)

    with driver.session(database=NEO4J_DATABASE) as session:
        # 1. Dem chunks
        result = session.run("MATCH (c:MedicalChunk) RETURN count(c) as count")
        chunks = result.single()['count']
        print(f"MedicalChunks: {chunks}")

        # 2. Dem entities
        result = session.run("MATCH (e:Entity) RETURN count(e) as count")
        entities = result.single()['count']
        print(f"Entities: {entities}")

        # 3. Dem relationships
        result = session.run("MATCH ()-[r]->() RETURN count(r) as count")
        rels = result.single()['count']
        print(f"Relationships: {rels}")

        # 4. Kiem tra sample
        print("\nSample Entities:")
        result = session.run("""
            MATCH (e:Entity)
            RETURN e.name as name, e.type as type
            LIMIT 10
        """)
        for record in result:
            print(f"  {record['name']} ({record['type']})")

        print("\nSample Relationships:")
        result = session.run("""
            MATCH (s:Entity)-[r]->(t:Entity)
            RETURN s.name as source, type(r) as type, t.name as target
            LIMIT 10
        """)
        for record in result:
            print(f"  {record['source']} --({record['type']})--> {record['target']}")

# Chay kiem tra
verify_data(driver)

KIEM TRA DU LIEU TRONG NEO4J
MedicalChunks: 259
Entities: 55
Relationships: 53

Sample Entities:
  Sốt xuất huyết Dengue (benh)
  Paracetamol (thuoc)
  Sốt cao (trieu_chung)
  Truyền dịch (dich_truyen)
  Ringer lactate (thuoc)
  Ringer acetate (thuoc)
  NaCl 0,9% (thuoc)
  Hct (xet_nghiem)
  Tiểu cầu (xet_nghiem)
  AST/ALT (xet_nghiem)

Sample Relationships:
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Sốt cao
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Hct
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Sốt
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Hematocrit giảm
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Phù phổi cấp
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Đau đầu
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Xuất huyết
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Sốc
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Xuất huyết tiêu hóa
  Sốt xuất huyết Dengue --(CO_TRIEU_CHUNG)--> Tổn thương gan


In [ ]:
# ============================================
# CELL 15: TAO VA TEST VECTOR SEARCH (DA SUA)
# ============================================

import time
import hashlib
import pickle
import os
import getpass
from pathlib import Path
from neo4j import GraphDatabase
from langchain_google_genai import GoogleGenerativeAIEmbeddings

print("=" * 60)
print("TAO VA TEST VECTOR SEARCH")
print("=" * 60)

# ============================================
# 1. LAY THONG TIN TU get_secret
# ============================================

print("\n[0] LAY THONG TIN NEO4J VA API")
print("-" * 40)

def get_secret(secret_name: str):
    """Lấy secret từ Google Colab nếu có; nếu không, yêu cầu nhập thủ công."""
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(secret_name)
    except Exception:
        value = os.environ.get(secret_name)
    if not value:
        value = getpass.getpass(f"Nhập {secret_name}: ")
    return value

# Lay thong tin
GOOGLE_API_KEY = get_secret("APIDengue")
NEO4J_URI = get_secret("NEO4J_URI")
NEO4J_USERNAME = get_secret("NEO4J_USERNAME")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE")

print("Da lay thong tin tu Secret:")
print(f"  GOOGLE_API_KEY: {'*' * 8}" if GOOGLE_API_KEY else "  GOOGLE_API_KEY: Khong tim thay")
print(f"  NEO4J_URI: {NEO4J_URI[:30]}..." if NEO4J_URI else "  NEO4J_URI: Khong tim thay")
print(f"  NEO4J_USERNAME: {NEO4J_USERNAME}" if NEO4J_USERNAME else "  NEO4J_USERNAME: Khong tim thay")
print(f"  NEO4J_PASSWORD: {'*' * 8}" if NEO4J_PASSWORD else "  NEO4J_PASSWORD: Khong tim thay")
print(f"  NEO4J_DATABASE: {NEO4J_DATABASE}" if NEO4J_DATABASE else "  NEO4J_DATABASE: Khong tim thay")

# Kiem tra thong tin
if not NEO4J_URI or not NEO4J_USERNAME or not NEO4J_PASSWORD:
    print("\nTHIEU THONG TIN NEO4J!")
    raise ValueError("Thieu thong tin ket noi Neo4j")

# ============================================
# 2. KET NOI NEO4J
# ============================================

print("\n[1] KET NOI NEO4J")
print("-" * 40)

try:
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
    )
    driver.verify_connectivity()
    print("Ket noi Neo4j thanh cong!")

    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run("RETURN 1 as test")
        print(f"Database: {NEO4J_DATABASE} - OK")

except Exception as e:
    print(f"LOI ket noi: {e}")
    raise

# ============================================
# 3. KHOI TAO EMBEDDING
# ============================================

print("\n[2] KHOI TAO EMBEDDING MODEL")
print("-" * 40)

try:
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=GOOGLE_API_KEY,
        output_dimensionality=768
    )
    print("Da khoi tao embedding model")

    test_embedding = embeddings.embed_query("Test")
    print(f"Embedding dimension: {len(test_embedding)}")

except Exception as e:
    print(f"LOI khoi tao embedding: {e}")
    raise

# ============================================
# 4. KIEM TRA VA TAO VECTOR INDEX
# ============================================

print("\n[3] KIEM TRA VA TAO VECTOR INDEX")
print("-" * 40)

def check_and_create_vector_index(driver):
    """Kiem tra va tao vector index neu chua co"""

    with driver.session(database=NEO4J_DATABASE) as session:
        try:
            result = session.run("""
                SHOW INDEXES YIELD name, type, state
                WHERE type = 'VECTOR'
                RETURN name, state
            """)
            indexes = list(result)

            if indexes:
                for idx in indexes:
                    if idx['name'] == 'entity_embeddings':
                        print(f"Vector index 'entity_embeddings' da ton tai, trang thai: {idx['state']}")
                        if idx['state'] == 'ONLINE':
                            return True
                        else:
                            print("Index dang o trang thai khong ONLINE, se tao lai...")
            else:
                print("Chua co vector index nao, se tao moi...")

        except Exception as e:
            print(f"Loi kiem tra index: {e}")
            print("Se thu tao index moi...")

        print("Dang tao vector index 'entity_embeddings'...")

        try:
            try:
                session.run("DROP INDEX entity_embeddings IF EXISTS")
                print("Da xoa index cu (neu co)")
            except Exception as e:
                print(f"Khong the xoa index: {e}")

            session.run("""
                CREATE VECTOR INDEX entity_embeddings IF NOT EXISTS
                FOR (e:Entity) ON (e.embedding)
                OPTIONS {indexConfig: {
                    `vector.dimensions`: 768,
                    `vector.similarity_function`: 'cosine'
                }}
            """)
            print("Da tao vector index thanh cong!")

            result = session.run("""
                SHOW INDEXES YIELD name, state
                WHERE name = 'entity_embeddings'
                RETURN state
            """)
            record = result.single()
            if record:
                print(f"Trang thai index: {record['state']}")
                return True
            else:
                print("Khong tim thay index sau khi tao")
                return False

        except Exception as e:
            print(f"LOI tao vector index: {e}")
            return False

index_created = check_and_create_vector_index(driver)

# ============================================
# 5. TAO EMBEDDING CHO ENTITIES
# ============================================

print("\n[4] TAO EMBEDDING CHO ENTITIES")
print("-" * 40)

def create_embeddings_for_entities(driver, embeddings, batch_size=10):
    """Tao embedding cho entities chua co embedding"""

    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run("MATCH (e:Entity) RETURN count(e) as total")
        total_entities = result.single()['total']
        print(f"Tong entities: {total_entities}")

        result = session.run("""
            MATCH (e:Entity)
            WHERE e.embedding IS NULL
            RETURN e.id as id, e.name as name, e.type as type
        """)

        entities_without_embedding = list(result)
        print(f"Entities chua co embedding: {len(entities_without_embedding)}")

        if not entities_without_embedding:
            print("Tat ca entities da co embedding!")
            return True

        total_batches = (len(entities_without_embedding) + batch_size - 1) // batch_size

        print(f"\nBat dau tao embedding cho {len(entities_without_embedding)} entities...")

        for batch_idx in range(total_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(entities_without_embedding))
            batch = entities_without_embedding[start_idx:end_idx]

            print(f"\nBatch {batch_idx + 1}/{total_batches}: {len(batch)} entities")

            for i, entity in enumerate(batch):
                try:
                    entity_name = entity['name']
                    entity_type = entity['type']

                    text_to_embed = f"{entity_name} {entity_type}"
                    embedding = embeddings.embed_query(text_to_embed)

                    session.run("""
                        MATCH (e:Entity {id: $id})
                        SET e.embedding = $embedding,
                            e.updated_at = datetime()
                        """,
                        id=entity['id'],
                        embedding=embedding
                    )

                    if (i + 1) % 5 == 0 or (i + 1) == len(batch):
                        print(f"  Da tao embedding cho {i + 1}/{len(batch)} entities")

                except Exception as e:
                    print(f"  LOI tao embedding cho {entity['name']}: {e}")
                    continue

                time.sleep(0.3)

            if batch_idx < total_batches - 1:
                print("  Cho 5s truoc batch tiep theo...")
                time.sleep(5)

        print(f"\nDa tao embedding cho {len(entities_without_embedding)} entities")

        result = session.run("""
            MATCH (e:Entity)
            WHERE e.embedding IS NOT NULL
            RETURN count(e) as count
        """)
        count = result.single()['count']
        print(f"Tong entities co embedding: {count}/{total_entities}")

        return True

if index_created:
    embedding_created = create_embeddings_for_entities(driver, embeddings, batch_size=10)

# ============================================
# 6. CAC HAM SEARCH (FIXED)
# ============================================

print("\n[5] TAO CAC HAM SEARCH")
print("-" * 40)

def vector_search_fixed(driver, query, embeddings, top_k=5):
    """
    Tim kiem vector trong Neo4j - DA FIX
    """
    query_embedding = embeddings.embed_query(query)

    with driver.session(database=NEO4J_DATABASE) as session:
        try:
            embedding_list = list(query_embedding)

            result = session.run("""
                CALL db.index.vector.queryNodes('entity_embeddings', $top_k, $embedding)
                YIELD node, score
                RETURN node.name AS name,
                       node.type AS type,
                       node.id AS id,
                       score
                ORDER BY score DESC
            """,
            top_k=top_k,
            embedding=embedding_list
            )

            results = []
            for record in result:
                results.append({
                    'name': record['name'],
                    'type': record['type'],
                    'id': record['id'],
                    'score': round(record['score'], 4)
                })

            return results

        except Exception as e:
            print(f"LOI vector search: {e}")
            return fallback_search_by_name(driver, query, top_k)

def fallback_search_by_name(driver, query, top_k=5):
    """
    Fallback: Tim kiem bang ten entity gan giong
    """
    with driver.session(database=NEO4J_DATABASE) as session:
        keywords = query.lower().split()

        results = []
        for keyword in keywords[:3]:
            result = session.run("""
                MATCH (e:Entity)
                WHERE e.name CONTAINS $keyword
                RETURN e.name AS name,
                       e.type AS type,
                       e.id AS id
                LIMIT $limit
            """, keyword=keyword, limit=top_k)

            for record in result:
                results.append({
                    'name': record['name'],
                    'type': record['type'],
                    'id': record['id'],
                    'score': 0.5
                })

        seen = set()
        unique_results = []
        for r in results:
            key = (r['name'], r['type'])
            if key not in seen:
                seen.add(key)
                unique_results.append(r)

        return unique_results[:top_k]

def graph_search_fixed(driver, entity_name, max_depth=2, limit=20):
    """
    Tim kiem theo do thi tu mot entity - DA FIX
    """
    with driver.session(database=NEO4J_DATABASE) as session:
        if max_depth < 1:
            max_depth = 1
        elif max_depth > 3:
            max_depth = 3

        query = f"""
            MATCH path = (e:Entity {{name: $entity_name}})-[*1..{max_depth}]-(related)
            WHERE e <> related
            RETURN
                related.name AS name,
                related.type AS type,
                related.id AS id,
                length(path) AS distance,
                [rel IN relationships(path) | type(rel)] AS relationship_types
            LIMIT $limit
        """

        result = session.run(
            query,
            entity_name=entity_name,
            limit=limit
        )

        results = []
        for record in result:
            results.append({
                'name': record['name'],
                'type': record['type'],
                'id': record['id'],
                'distance': record['distance'],
                'relationships': record['relationship_types']
            })

        return results

def hybrid_search_fixed(driver, query, embeddings, top_k=5, max_depth=2):
    """
    Ket hop vector search va graph traversal - DA FIX
    """
    print(f"\nHYBRID SEARCH: '{query}'")
    print("-" * 40)

    print("Vector search...")
    vector_results = vector_search_fixed(driver, query, embeddings, top_k)

    if not vector_results:
        print("Khong co ket qua vector search, thu fallback...")
        vector_results = fallback_search_by_name(driver, query, top_k)

    print(f"Vector search: {len(vector_results)} results")
    for r in vector_results:
        print(f"   - {r['name']} ({r['type']}) - score: {r.get('score', 'N/A')}")

    print("\nGraph traversal...")
    all_graph_results = []
    for r in vector_results[:3]:
        entity_name = r['name']
        print(f"   Traverse tu: {entity_name}")
        graph_results = graph_search_fixed(driver, entity_name, max_depth, limit=10)
        all_graph_results.extend(graph_results)
        print(f"      -> Tim thay {len(graph_results)} entities lien quan")

    seen = set()
    unique_graph_results = []
    for r in all_graph_results:
        key = r['name']
        if key not in seen:
            seen.add(key)
            unique_graph_results.append(r)

    print(f"Graph results (sau dedup): {len(unique_graph_results)}")

    entity_names = [r['name'] for r in vector_results]
    related_chunks = []

    if entity_names:
        print("\nTim chunks lien quan...")
        with driver.session(database=NEO4J_DATABASE) as session:
            for entity_name in entity_names[:3]:
                result = session.run("""
                    MATCH (e:Entity {name: $entity_name})-[r]-(c:MedicalChunk)
                    WHERE type(r) IN ['BELONGS_TO', 'FROM_CHUNK', 'REFERENCES']
                    RETURN c.text AS text,
                           c.page AS page,
                           c.type AS type,
                           e.name AS entity_name
                    LIMIT 3
                """, entity_name=entity_name)

                for record in result:
                    related_chunks.append({
                        'text': record['text'][:500] + "..." if len(record['text']) > 500 else record['text'],
                        'page': record['page'],
                        'type': record['type'],
                        'entity': record['entity_name']
                    })

    print(f"Found {len(related_chunks)} related chunks")

    return {
        'query': query,
        'vector_results': vector_results,
        'graph_results': unique_graph_results[:10],
        'related_chunks': related_chunks
    }

print("Da tao cac ham search")

# ============================================
# 7. TEST HYBRID SEARCH
# ============================================

print("\n[6] TEST HYBRID SEARCH")
print("-" * 40)

test_query = "Phac do dieu tri soc sot xuat huyet Dengue"

try:
    results = hybrid_search_fixed(driver, test_query, embeddings, top_k=5)

    print("\n" + "=" * 60)
    print("KET QUA HYBRID SEARCH")
    print("=" * 60)

    print(f"\nQuery: {results['query']}")

    print(f"\nVector Results ({len(results['vector_results'])}):")
    for r in results['vector_results']:
        print(f"   - {r['name']} ({r['type']}) - score: {r.get('score', 'N/A')}")

    print(f"\nGraph Results ({len(results['graph_results'])}):")
    for r in results['graph_results'][:5]:
        print(f"   - {r['name']} ({r['type']}) - distance: {r['distance']}")

    print(f"\nRelated Chunks ({len(results['related_chunks'])}):")
    for r in results['related_chunks'][:3]:
        print(f"   - Page {r['page']}: {r['text'][:150]}...")

except Exception as e:
    print(f"LOI: {e}")
    import traceback
    traceback.print_exc()

# ============================================
# 8. TEST VECTOR SEARCH
# ============================================

print("\n[7] TEST VECTOR SEARCH")
print("-" * 40)

test_queries = [
    "Dieu tri sot xuat huyet",
    "Trieu chung sot xuat huyet"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    results = vector_search_fixed(driver, query, embeddings, top_k=5)

    if results:
        print("Ket qua vector search:")
        for i, r in enumerate(results, 1):
            print(f"  {i}. {r['name']} ({r['type']}) - score: {r['score']}")
    else:
        print("  Khong co ket qua")
    print("-" * 40)

# ============================================
# 9. LUU CAC HAM (KHONG LUU DRIVER)
# ============================================

print("\n[8] LUU CAC HAM DE SU DUNG")
print("-" * 40)

# Chi luu cac ham va thong tin, khong luu driver
retrieval_functions = {
    'vector_search': vector_search_fixed,
    'graph_search': graph_search_fixed,
    'hybrid_search': hybrid_search_fixed,
    'fallback_search': fallback_search_by_name,
    'NEO4J_URI': NEO4J_URI,
    'NEO4J_USERNAME': NEO4J_USERNAME,
    'NEO4J_PASSWORD': NEO4J_PASSWORD,
    'NEO4J_DATABASE': NEO4J_DATABASE,
    'EMBEDDING_MODEL': "models/gemini-embedding-001",
    'EMBEDDING_DIMENSION': 768
}

with open('retrieval_functions.pkl', 'wb') as f:
    pickle.dump(retrieval_functions, f)

print("Da luu cac ham vao file 'retrieval_functions.pkl'")

# ============================================
# 10. HUONG DAN SU DUNG
# ============================================

print("\n" + "=" * 60)
print("HOAN TAT! VECTOR SEARCH DA SAN SANG")
print("=" * 60)

print("""
CACH SU DUNG:

1. Load cac ham:
   import pickle
   from neo4j import GraphDatabase
   from langchain_google_genai import GoogleGenerativeAIEmbeddings
   from google.colab import userdata

   with open('retrieval_functions.pkl', 'rb') as f:
       funcs = pickle.load(f)

   vector_search = funcs['vector_search']
   graph_search = funcs['graph_search']
   hybrid_search = funcs['hybrid_search']

   # Ket noi Neo4j
   driver = GraphDatabase.driver(
       funcs['NEO4J_URI'],
       auth=(funcs['NEO4J_USERNAME'], funcs['NEO4J_PASSWORD'])
   )

   # Khoi tao embedding
   GOOGLE_API_KEY = userdata.get('APIDengue')
   embeddings = GoogleGenerativeAIEmbeddings(
       model=funcs['EMBEDDING_MODEL'],
       google_api_key=GOOGLE_API_KEY,
       output_dimensionality=funcs['EMBEDDING_DIMENSION']
   )

2. Su dung vector search:
   results = vector_search(driver, "Cau hoi cua ban", embeddings, top_k=5)
   for r in results:
       print(f"{r['name']} ({r['type']}) - {r['score']}")

3. Su dung graph search:
   results = graph_search(driver, "Sot xuat huyet Dengue", max_depth=2)

4. Su dung hybrid search:
   results = hybrid_search(driver, "Cau hoi", embeddings, top_k=5)
""")

print("\n" + "=" * 60)

TAO VA TEST VECTOR SEARCH

[0] LAY THONG TIN NEO4J VA API
----------------------------------------
Da lay thong tin tu Secret:
  GOOGLE_API_KEY: ********
  NEO4J_URI: neo4j+s://1db0cccb.databases.n...
  NEO4J_USERNAME: 1db0cccb
  NEO4J_PASSWORD: ********
  NEO4J_DATABASE: 1db0cccb

[1] KET NOI NEO4J
----------------------------------------
Ket noi Neo4j thanh cong!
Database: 1db0cccb - OK

[2] KHOI TAO EMBEDDING MODEL
----------------------------------------
Da khoi tao embedding model
Embedding dimension: 768

[3] KIEM TRA VA TAO VECTOR INDEX
----------------------------------------
Vector index 'entity_embeddings' da ton tai, trang thai: ONLINE

[4] TAO EMBEDDING CHO ENTITIES
----------------------------------------
Tong entities: 55
Entities chua co embedding: 0
Tat ca entities da co embedding!

[5] TAO CAC HAM SEARCH
----------------------------------------
Da tao cac ham search

[6] TEST HYBRID SEARCH
----------------------------------------

HYBRID SEARCH: 'Phac do dieu tri soc s

Vector search: 5 results
   - Sốt xuất huyết Dengue (benh) - score: 0.895
   - Truyền máu (dieu_tri) - score: 0.8536
   - Xuất huyết (trieu_chung) - score: 0.8503
   - Truyền dịch (dieu_tri) - score: 0.8496
   - Truyền máu sớm (dieu_tri) - score: 0.8411

Graph traversal...
   Traverse tu: Sốt xuất huyết Dengue
      -> Tim thay 10 entities lien quan
   Traverse tu: Truyền máu
      -> Tim thay 10 entities lien quan
   Traverse tu: Xuất huyết
      -> Tim thay 10 entities lien quan
Graph results (sau dedup): 13

Tim chunks lien quan...
Found 0 related chunks

KET QUA HYBRID SEARCH

Query: Phac do dieu tri soc sot xuat huyet Dengue

Vector Results (5):
   - Sốt xuất huyết Dengue (benh) - score: 0.895
   - Truyền máu (dieu_tri) - score: 0.8536
   - Xuất huyết (trieu_chung) - score: 0.8503
   - Truyền dịch (dieu_tri) - score: 0.8496
   - Truyền máu sớm (dieu_tri) - score: 0.8411

Graph Results (10):
   - Sốt cao (trieu_chung) - distance: 1
   - Hct (xet_nghiem) - distance: 1
   - Sốt (trie

Ket qua vector search:
  1. Xuất huyết (trieu_chung) - score: 0.8826
  2. Xuất huyết tiêu hóa (trieu_chung) - score: 0.8714
  3. Truyền máu (dieu_tri) - score: 0.8657
  4. Truyền dịch (dieu_tri) - score: 0.8527
  5. Truyền máu sớm (dieu_tri) - score: 0.8507
----------------------------------------

Query: Trieu chung sot xuat huyet


Ket qua vector search:
  1. Xuất huyết (trieu_chung) - score: 0.941
  2. Sốt cao (trieu_chung) - score: 0.9273
  3. Sốt (trieu_chung) - score: 0.9222
  4. Xuất huyết tiêu hóa (trieu_chung) - score: 0.9178
  5. Hematocrit giảm (trieu_chung) - score: 0.8874
----------------------------------------

[8] LUU CAC HAM DE SU DUNG
----------------------------------------
Da luu cac ham vao file 'retrieval_functions.pkl'

HOAN TAT! VECTOR SEARCH DA SAN SANG

CACH SU DUNG:

1. Load cac ham:
   import pickle
   from neo4j import GraphDatabase
   from langchain_google_genai import GoogleGenerativeAIEmbeddings
   from google.colab import userdata

   with open('retrieval_functions.pkl', 'rb') as f:
       funcs = pickle.load(f)

   vector_search = funcs['vector_search']
   graph_search = funcs['graph_search']
   hybrid_search = funcs['hybrid_search']

   # Ket noi Neo4j
   driver = GraphDatabase.driver(
       funcs['NEO4J_URI'],
       auth=(funcs['NEO4J_USERNAME'], funcs['NEO4J_PASSWORD'])
   )



In [ ]:
# ============================================
# CELL 15.1: LUU CAC HAM (KHONG LUU DRIVER)
# ============================================

print("LUU CAC HAM DE SU DUNG")
print("=" * 60)

# Tao dict chua cac ham (khong luu driver vi khong pickle duoc)
retrieval_functions = {
    'vector_search': vector_search,
    'graph_search': graph_search,
    'hybrid_search': hybrid_search,
    'NEO4J_URI': NEO4J_URI,
    'NEO4J_USERNAME': NEO4J_USERNAME,
    'NEO4J_PASSWORD': NEO4J_PASSWORD,
    'NEO4J_DATABASE': NEO4J_DATABASE,
    'EMBEDDING_MODEL': "models/gemini-embedding-001",
    'EMBEDDING_DIMENSION': 768
}

# Luu vao file
import pickle
with open('retrieval_functions.pkl', 'wb') as f:
    pickle.dump(retrieval_functions, f)

print("Da luu cac ham vao file 'retrieval_functions.pkl'")

# ============================================
# HUONG DAN SU DUNG
# ============================================

print("\n" + "=" * 60)
print("HUONG DAN SU DUNG VECTOR SEARCH")
print("=" * 60)

print("""
# 1. Load cac ham da luu
import pickle
from neo4j import GraphDatabase
from langchain_google_genai import GoogleGenerativeAIEmbeddings

with open('retrieval_functions.pkl', 'rb') as f:
    funcs = pickle.load(f)

# Lay cac ham
vector_search = funcs['vector_search']
graph_search = funcs['graph_search']
hybrid_search = funcs['hybrid_search']

# Lay thong tin ket noi
NEO4J_URI = funcs['NEO4J_URI']
NEO4J_USERNAME = funcs['NEO4J_USERNAME']
NEO4J_PASSWORD = funcs['NEO4J_PASSWORD']
NEO4J_DATABASE = funcs['NEO4J_DATABASE']

# Ket noi Neo4j
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Khoi tao embedding
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('APIDengue')
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
    output_dimensionality=768
)

# 2. Vector search
query = "Cach dieu tri sot xuat huyet"
results = vector_search(driver, query, embeddings, top_k=5)

print(f"Ket qua cho: {query}")
for r in results:
    print(f"  - {r['name']} ({r['type']}) - score: {r['score']}")

# 3. Graph search
results = graph_search(driver, "Sot xuat huyet Dengue", max_depth=2)
for r in results:
    print(f"  - {r['name']} ({r['type']}) - cach {r['distance']} hop")

# 4. Hybrid search
results = hybrid_search(driver, query, embeddings, top_k=5)
print(f"Vector: {len(results['vector_results'])}")
print(f"Graph: {len(results['graph_results'])}")
print(f"Chunks: {len(results['related_chunks'])}")
""")

print("\n" + "=" * 60)
print("VECTOR SEARCH HOAN TAT!")
print("=" * 60)

LUU CAC HAM DE SU DUNG


NameError: name 'vector_search' is not defined

In [ ]:
# ============================================
# TEST VECTOR SEARCH
# ============================================

query = "sot xuat huyet"

print(f"Tim kiem: '{query}'")
print("-" * 40)

# Tao embedding
query_embedding = embeddings.embed_query(query)

# Vector search
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("""
        CALL db.index.vector.queryNodes('entity_embeddings', 5, $embedding)
        YIELD node, score
        RETURN node.name AS name, node.type AS type, score
        ORDER BY score DESC
    """, embedding=list(query_embedding))

    print("Ket qua:")
    for i, record in enumerate(result, 1):
        print(f"{i}. {record['name']} ({record['type']}) ")

Tim kiem: 'sot xuat huyet'
----------------------------------------


Ket qua:
1. Xuất huyết (trieu_chung) 
2. Sốt cao (trieu_chung) 
3. Sốt (trieu_chung) 
4. Sốt xuất huyết Dengue (benh) 
5. Xuất huyết tiêu hóa (trieu_chung) 


In [ ]:
# ============================================
# CELL 16: TRIEN KHAI RAG VOI LLAMA  (GROQ) - CHI HIEN THI CAU HOI VA CAU TRA LOI
# ============================================

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from IPython.display import Markdown, display
import hashlib
import json
import time
from typing import List, Dict, Any, Optional

print("=" * 60)
print("TRIEN KHAI RAG VOI LLAMA ")
print("=" * 60)

# ============================================
# 1. CAU HINH LLM VOI GROQ
# ============================================

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')

    if not GROQ_API_KEY:
        import getpass
        GROQ_API_KEY = getpass.getpass("Nhap GROQ_API_KEY: ")

    CHAT_MODEL = "openai/gpt-oss-20b"
    llm = ChatGroq(
        model=CHAT_MODEL,
        temperature=0.2,
        groq_api_key=GROQ_API_KEY,
        max_tokens=1024
    )

    print(f"Da khoi tao thanh cong voi Groq model: {CHAT_MODEL}")

    test_response = llm.invoke("Xin chao, hay gioi thieu ngan gon ve ban")
    print(f"Test thanh cong: {test_response.content[:100]}...")

except Exception as e:
    print(f"Loi khoi tao Groq: {e}")
    raise



def graph_search_fixed(driver, entity_name, max_depth=2, limit=20):
    """
    Tim kiem theo do thi tu mot entity - DA FIX VOI MAX_DEPTH
    """
    with driver.session(database=NEO4J_DATABASE) as session:
        if max_depth < 1:
            max_depth = 1
        elif max_depth > 3:
            max_depth = 3

        query = f"""
            MATCH path = (e:Entity {{name: $entity_name}})-[*1..{max_depth}]-(related)
            WHERE e <> related
            RETURN
                related.name AS name,
                related.type AS type,
                related.id AS id,
                length(path) AS distance,
                [rel IN relationships(path) | type(rel)] AS relationship_types
            LIMIT $limit
        """

        result = session.run(
            query,
            entity_name=entity_name,
            limit=limit
        )

        results = []
        for record in result:
            results.append({
                'name': record['name'],
                'type': record['type'],
                'id': record['id'],
                'distance': record['distance'],
                'relationships': record['relationship_types']
            })

        return results

# ============================================
# 3. SUA HAM HYBRID_SEARCH
# ============================================

def hybrid_search_fixed(driver, query, embeddings, top_k=5, max_depth=2):
    """
    Ket hop vector search va graph traversal - DA FIX
    """
    print(f"\nHYBRID SEARCH: '{query}'")
    print("-" * 40)

    vector_results = vector_search_fixed(driver, query, embeddings, top_k)

    if not vector_results:
        print("Khong co ket qua vector search, thu fallback...")
        vector_results = fallback_search_by_name(driver, query, top_k)

    print(f"Vector search: {len(vector_results)} results")
    for r in vector_results:
        print(f"   - {r['name']} ({r['type']}) - score: {r.get('score', 'N/A')}")

    print("\nGraph traversal...")
    all_graph_results = []
    for r in vector_results[:3]:
        entity_name = r['name']
        print(f"   Traverse tu: {entity_name}")
        graph_results = graph_search_fixed(driver, entity_name, max_depth, limit=10)
        all_graph_results.extend(graph_results)
        print(f"      -> Tim thay {len(graph_results)} entities lien quan")

    seen = set()
    unique_graph_results = []
    for r in all_graph_results:
        key = r['name']
        if key not in seen:
            seen.add(key)
            unique_graph_results.append(r)

    print(f"Graph results (sau dedup): {len(unique_graph_results)}")

    entity_names = [r['name'] for r in vector_results]
    related_chunks = []

    if entity_names:
        print("\nTim chunks lien quan...")
        with driver.session(database=NEO4J_DATABASE) as session:
            for entity_name in entity_names[:3]:
                result = session.run("""
                    MATCH (e:Entity {name: $entity_name})-[r]-(c:MedicalChunk)
                    WHERE type(r) IN ['BELONGS_TO', 'FROM_CHUNK', 'REFERENCES']
                    RETURN c.text AS text,
                           c.page AS page,
                           c.type AS type,
                           e.name AS entity_name
                    LIMIT 3
                """, entity_name=entity_name)

                for record in result:
                    related_chunks.append({
                        'text': record['text'][:500] + "..." if len(record['text']) > 500 else record['text'],
                        'page': record['page'],
                        'type': record['type'],
                        'entity': record['entity_name']
                    })

    print(f"Found {len(related_chunks)} related chunks")

    return {
        'query': query,
        'vector_results': vector_results,
        'graph_results': unique_graph_results[:10],
        'related_chunks': related_chunks
    }

# ============================================
# 4. SUA HAM VECTOR_SEARCH
# ============================================

def vector_search_fixed(driver, query, embeddings, top_k=5):
    """
    Tim kiem vector trong Neo4j - DA FIX
    """
    query_embedding = embeddings.embed_query(query)

    with driver.session(database=NEO4J_DATABASE) as session:
        try:
            embedding_list = list(query_embedding)

            result = session.run("""
                CALL db.index.vector.queryNodes('entity_embeddings', $top_k, $embedding)
                YIELD node, score
                RETURN node.name AS name,
                       node.type AS type,
                       node.id AS id,
                       score
                ORDER BY score DESC
            """,
            top_k=top_k,
            embedding=embedding_list
            )

            results = []
            for record in result:
                results.append({
                    'name': record['name'],
                    'type': record['type'],
                    'id': record['id'],
                    'score': round(record['score'], 4)
                })

            return results

        except Exception as e:
            print(f"LOI vector search: {e}")
            return fallback_search_by_name(driver, query, top_k)

def fallback_search_by_name(driver, query, top_k=5):
    """
    Fallback: Tim kiem bang ten entity gan giong
    """
    with driver.session(database=NEO4J_DATABASE) as session:
        keywords = query.lower().split()

        results = []
        for keyword in keywords[:3]:
            result = session.run("""
                MATCH (e:Entity)
                WHERE e.name CONTAINS $keyword
                RETURN e.name AS name,
                       e.type AS type,
                       e.id AS id
                LIMIT $limit
            """, keyword=keyword, limit=top_k)

            for record in result:
                results.append({
                    'name': record['name'],
                    'type': record['type'],
                    'id': record['id'],
                    'score': 0.5
                })

        seen = set()
        unique_results = []
        for r in results:
            key = (r['name'], r['type'])
            if key not in seen:
                seen.add(key)
                unique_results.append(r)

        return unique_results[:top_k]

# ============================================
# 5. HAM RETRIEVE TU NEO4J - SUA LAI
# ============================================

def retrieve_from_neo4j(
    driver,
    query: str,
    embeddings,
    top_k: int = 5,
    use_hybrid: bool = True,
    max_depth: int = 2
) -> Dict:
    """
    Truy hoi du lieu tu Neo4j bang hybrid search
    """

    print(f"\n[Retrieval] Query: '{query}'")
    print("-" * 40)

    if use_hybrid:
        results = hybrid_search_fixed(driver, query, embeddings, top_k=top_k, max_depth=max_depth)

        return {
            "entities": results.get('vector_results', []),
            "related_entities": results.get('graph_results', []),
            "chunks": results.get('related_chunks', []),
            "query": query,
            "num_entities": len(results.get('vector_results', [])),
            "num_related": len(results.get('graph_results', [])),
            "num_chunks": len(results.get('related_chunks', []))
        }
    else:
        entities = vector_search_fixed(driver, query, embeddings, top_k=top_k)

        chunks = []
        with driver.session(database=NEO4J_DATABASE) as session:
            for entity in entities[:3]:
                result = session.run("""
                    MATCH (e:Entity {name: $entity_name})-[r]-(c:MedicalChunk)
                    WHERE type(r) IN ['BELONGS_TO', 'FROM_CHUNK', 'REFERENCES']
                    RETURN c.text AS text,
                           c.page AS page,
                           c.type AS type
                    LIMIT 3
                """, entity_name=entity['name'])

                for record in result:
                    chunks.append({
                        'text': record['text'],
                        'page': record['page'],
                        'type': record['type'],
                        'entity': entity['name']
                    })

        return {
            "entities": entities,
            "related_entities": [],
            "chunks": chunks,
            "query": query,
            "num_entities": len(entities),
            "num_related": 0,
            "num_chunks": len(chunks)
        }

# ============================================
# 6. HAM FORMAT CONTEXT
# ============================================

def format_context_for_rag(retrieval_results: Dict) -> str:
    """
    Format ket qua truy hoi thanh context cho LLM
    """
    context_parts = []

    if retrieval_results.get('entities'):
        context_parts.append("=== CAC ENTITY LIEN QUAN ===")
        for i, entity in enumerate(retrieval_results['entities'][:10], 1):
            score = entity.get('score', 'N/A')
            context_parts.append(f"{i}. {entity['name']} (Loai: {entity['type']}) - Do tuong dong: {score}")

    if retrieval_results.get('related_entities'):
        context_parts.append("\n=== CAC ENTITY LIEN QUAN QUA QUAN HE ===")
        for i, entity in enumerate(retrieval_results['related_entities'][:8], 1):
            dist = entity.get('distance', 'N/A')
            rels = entity.get('relationships', [])
            rel_str = " -> ".join(rels[:3]) if rels else "khong ro"
            context_parts.append(f"{i}. {entity['name']} (Loai: {entity['type']}) - Cach {dist} buoc, Quan he: {rel_str}")

    if retrieval_results.get('chunks'):
        context_parts.append("\n=== DOAN VAN LIEN QUAN ===")
        for i, chunk in enumerate(retrieval_results['chunks'][:5], 1):
            text = chunk.get('text', '')[:600]
            page = chunk.get('page', 'N/A')
            entity = chunk.get('entity', 'khong ro')
            context_parts.append(f"--- Doan {i} (Trang {page}, Entity: {entity}) ---")
            context_parts.append(text + ("..." if len(chunk.get('text', '')) > 600 else ""))
            context_parts.append("")

    if not context_parts:
        return "Không tìm thấy thông tin về Sốt xuất huyết Dengue trong cơ sở dữ liệu."

    return "\n".join(context_parts)

# ============================================
# 7. SYSTEM PROMPT CHO RAG
# ============================================

SYSTEM_PROMPT = """Bạn là trợ lý y tế chuyên về bệnh Sốt xuất huyết Dengue. Sử dụng các thông tin được cung cấp dưới đây để trả lời câu hỏi.

YÊU CẦU:
- Chỉ sử dụng thông tin từ CONTEXT được cung cấp
- Nếu câu trả lời không có trong CONTEXT, hãy nói rõ "Không có thông tin này trong tài liệu"
- Trả lời bằng tiếng Việt, ngắn gọn, rõ ràng
- Nếu có nhiều thông tin, hãy tóm tắt thành 3-5 câu
- Ưu tiên thông tin về: chuẩn đoán, điều trị, triệu chứng, thuốc, liều dùng, xét nghiệm

CONTEXT:
{context}

-- ĐANG XỬ LÝ --
"""

# ============================================
# 8. HAM RAG CHINH
# ============================================

def rag_answer(
    user_query: str,
    driver,
    embeddings,
    k: int = 5,
    use_hybrid: bool = True,
    max_depth: int = 2,
    show_sources: bool = True
) -> Dict:
    """
    Ham RAG chinh de tra loi cau hoi
    """

    print(f"\nCÂU HỎI: {user_query}")

    retrieval_results = retrieve_from_neo4j(
        driver=driver,
        query=user_query,
        embeddings=embeddings,
        top_k=k,
        use_hybrid=use_hybrid,
        max_depth=max_depth
    )

    context = format_context_for_rag(retrieval_results)

    if retrieval_results['num_entities'] == 0 and retrieval_results['num_chunks'] == 0:
        return {
            "question": user_query,
            "answer": "Khong tim thay thong tin lien quan ve Sot xuat huyet Dengue trong co so du lieu.",
            "num_retrieved_entities": 0,
            "num_retrieved_chunks": 0,
            "sources": [],
            "context": context
        }

    try:
        messages = [
            SystemMessage(content=SYSTEM_PROMPT.format(context=context)),
            HumanMessage(content=user_query)
        ]

        response = llm.invoke(messages)
        answer = response.content.strip()

    except Exception as e:
        print(f"Loi goi LLM: {e}")
        answer = f"Loi khi goi LLM: {e}"

    sources = []
    for entity in retrieval_results.get('entities', []):
        if entity.get('name'):
            sources.append(f"Entity: {entity['name']} (Loai: {entity.get('type', 'unknown')})")

    for chunk in retrieval_results.get('chunks', []):
        page = chunk.get('page', 'N/A')
        entity = chunk.get('entity', 'unknown')
        sources.append(f"Trang {page} - Entity: {entity}")

    sources = list(dict.fromkeys(sources))

    return {
        "question": user_query,
        "answer": answer,
        "num_retrieved_entities": retrieval_results['num_entities'],
        "num_retrieved_chunks": retrieval_results['num_chunks'],
        "sources": sources[:10],
        "context": context,
        "retrieval_details": retrieval_results if show_sources else None,
        "model": CHAT_MODEL
    }

# ============================================
# 9. HAM HIEN THI KET QUA CHI CO CAU HOI VA CAU TRA LOI
# ============================================

def display_rag_result_simple(result: Dict):
    """Hien thi ket qua RAG chi co cau hoi va cau tra loi"""

    print("=" * 60)
    print(f"CAU HOI: {result['question']}")
    print("=" * 60)
    print(f"TRA LOI:\n{result['answer']}")
    print("=" * 60)

# ============================================
# 10. TEST RAG - CHI HIEN THI CAU HOI VA CAU TRA LOI
# ============================================

print("\n" + "=" * 60)
print("TEST RAG VỚI LLAMA ")
print("=" * 60)

if 'driver' not in globals() or driver is None:
    print("Chua co driver Neo4j, dang ket noi...")
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print("Da ket noi Neo4j")

if 'embeddings' not in globals() or embeddings is None:
    print("Chua co embeddings, dang khoi tao...")
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=GOOGLE_API_KEY,
        output_dimensionality=768
    )
    print("Da khoi tao embeddings")

# Danh sach cau hoi


print("\nHOÀN TẤT")

TRIEN KHAI RAG VOI LLAMA 
Da khoi tao thanh cong voi Groq model: openai/gpt-oss-20b
Test thanh cong: Chào bạn!  
Tôi là ChatGPT, một mô hình ngôn ngữ lớn được phát triển bởi OpenAI. Tôi có thể giúp bạn...

TEST RAG VỚI LLAMA 

HOÀN TẤT


In [ ]:
test_queries = [
    "sốt xuất huyết ở trẻ em có những biểu hiện nào?",
    "Sốt xuất huyết là gì? ",
    "Phác đồ điều trị của trẻ em"
]

for i, query in enumerate(test_queries, 1):
    print(f"\nTEST {i}:")

    try:
        result = rag_answer(
            user_query=query,
            driver=driver,
            embeddings=embeddings,
            k=5,
            use_hybrid=True,
            max_depth=2,
            show_sources=False
        )

        display_rag_result_simple(result)

    except Exception as e:
        print(f"Loi: {e}")

    if i < len(test_queries):
        time.sleep(2)


TEST 1:

CÂU HỎI: sốt xuất huyết ở trẻ em có những biểu hiện nào?

[Retrieval] Query: 'sốt xuất huyết ở trẻ em có những biểu hiện nào?'
----------------------------------------

HYBRID SEARCH: 'sốt xuất huyết ở trẻ em có những biểu hiện nào?'
----------------------------------------


Vector search: 5 results
   - Sốt xuất huyết Dengue (benh) - score: 0.8729
   - Sốt cao (trieu_chung) - score: 0.8525
   - Xuất huyết (trieu_chung) - score: 0.8451
   - Sốt (trieu_chung) - score: 0.8421
   - Chẩn đoán phân biệt nhiễm khuẩn huyết (trieu_chung) - score: 0.8331

Graph traversal...
   Traverse tu: Sốt xuất huyết Dengue
      -> Tim thay 10 entities lien quan
   Traverse tu: Sốt cao
      -> Tim thay 10 entities lien quan
   Traverse tu: Xuất huyết
      -> Tim thay 10 entities lien quan
Graph results (sau dedup): 13

Tim chunks lien quan...
Found 0 related chunks
CAU HOI: sốt xuất huyết ở trẻ em có những biểu hiện nào?
TRA LOI:
Sốt xuất huyết ở trẻ em thường biểu hiện với:

1. **Sốt cao** – nhiệt độ thường vượt 38 °C.  
2. **Đau đầu** – thường là triệu chứng đi kèm.  
3. **Xuất huyết** – có thể là chảy máu mũi, máu sữa, hoặc xuất huyết nội tạng.  
4. **Giảm hematocrit** – chỉ số Hct giảm, cho thấy suy giảm lượng hồng cầu.  

Những dấu hiệu này là những biểu hiện thường gặp

Vector search: 5 results
   - Sốt xuất huyết Dengue (benh) - score: 0.9009
   - Xuất huyết (trieu_chung) - score: 0.8649
   - Sốt (trieu_chung) - score: 0.8639
   - Sốt cao (trieu_chung) - score: 0.8584
   - Chẩn đoán phân biệt nhiễm khuẩn huyết (trieu_chung) - score: 0.8265

Graph traversal...
   Traverse tu: Sốt xuất huyết Dengue
      -> Tim thay 10 entities lien quan
   Traverse tu: Xuất huyết
      -> Tim thay 10 entities lien quan
   Traverse tu: Sốt
      -> Tim thay 10 entities lien quan
Graph results (sau dedup): 13

Tim chunks lien quan...
Found 0 related chunks
CAU HOI: Sốt xuất huyết là gì? 
TRA LOI:
Sốt xuất huyết là một bệnh do virus Dengue gây ra.  
Triệu chứng chính bao gồm sốt cao và xuất huyết.  
Ngoài ra, bệnh còn có thể gây đau đầu, giảm hematocrit và, trong trường hợp nặng, dẫn đến suy hô hấp hoặc sốc.

TEST 3:

CÂU HỎI: Phác đồ điều trị của trẻ em

[Retrieval] Query: 'Phác đồ điều trị của trẻ em'
----------------------------------------

HYBRID SEARCH: 'Phác đồ đi

Vector search: 5 results
   - Nhũ nhi < 1 tuổi hoặc dư cân (trieu_chung) - score: 0.8543
   - Đo và theo dõi (dieu_tri) - score: 0.8425
   - Truyền dịch (dieu_tri) - score: 0.8406
   - Nhập viện (dieu_tri) - score: 0.8375
   - Truyền máu (dieu_tri) - score: 0.829

Graph traversal...
   Traverse tu: Nhũ nhi < 1 tuổi hoặc dư cân
      -> Tim thay 10 entities lien quan
   Traverse tu: Đo và theo dõi
      -> Tim thay 10 entities lien quan
   Traverse tu: Truyền dịch
      -> Tim thay 10 entities lien quan
Graph results (sau dedup): 10

Tim chunks lien quan...
Found 0 related chunks
CAU HOI: Phác đồ điều trị của trẻ em
TRA LOI:
Không có thông tin này trong tài liệu.
